<a href="https://colab.research.google.com/github/xeonqq/FourierFeatureSiren/blob/main/ffn_vs_siren_and_combined.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Implicit Neural Functions


## Initialization

In [370]:
from typing import Any

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from torch.autograd import Variable
from PIL import Image, ImageFilter
import mon

torch.manual_seed(0)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [371]:
window_size   = 1
down_size     = 256
L    		  = 0.5
lr    	      = 1e-5
losses        = {}
psnrs         = {}
outputs       = {}
ress          = {}
total_steps   = 1000
summary_steps = 10
save_images   = True
font_size     = 10
line_width    = 2.0
fig_size      = (5, 4.5)
matplotlib.rc("font", **{
	# "family" : "normal",
	"size"   : font_size
})
plt.rcParams["figure.figsize"]    = [5, 4.5]
plt.rcParams["figure.autolayout"] = True

filename      = "a0952-kme_172"
image_file	  = f"data/zero_linr/{filename}_image.png"
depth_file    = f"data/zero_linr/{filename}_depth.png"
# ref_file      = f"data/zero_linr/{filename}_enhanced.png"
ref_file      = f"data/zero_linr/{filename}_ref.png"
output_dir    = f"run/zero_linr/{filename}"
image_v       = None
ref_v         = None
res_gt        = None

mon.delete_dir(output_dir)
mon.Path(output_dir).mkdir(parents=True, exist_ok=True)

## Misc

In [372]:
def mse(image1, image2):
	image_array1 = np.array(image1)
	image_array2 = np.array(image2)
	# Calculate the squared difference
	squared_difference = (image_array1 - image_array2) ** 2
	# Calculate the mean squared difference
	return np.mean(squared_difference)


def loss_to_psnr(loss, max=2):
	# return 10 * np.log10(max ** 2 / np.asarray(loss))
	return -10 * np.log10(2.0 * np.asarray(loss))


def laplace(y, x):
	grad = gradient(y, x)
	return divergence(grad, x)


def divergence(y, x):
	div = 0.
	for i in range(y.shape[-1]):
		div += torch.autograd.grad(y[..., i], x, torch.ones_like(y[..., i]), create_graph=True)[0][..., i:i+1]
	return div


def gradient(y, x, grad_outputs=None):
	if grad_outputs is None:
		grad_outputs = torch.ones_like(y)
	grad = torch.autograd.grad(y, [x], grad_outputs=grad_outputs, create_graph=True)[0]
	return grad

In [373]:
def get_image_tensor():
	image = get_image_color(image_file)
	depth = get_image_gray(depth_file)
	ref   = get_image_color(ref_file)
	return image, depth, ref


def get_image_color(path):
	"""Reads and returns RGB image, (1, 3, H, W)."""
	image = torch.from_numpy(np.array(Image.open(path))).float()
	image = image / torch.max(image)
	image = torch.movedim(image, -1, 0).unsqueeze(0)
	return image


def get_image_gray(path):
	"""Reads and returns RGB image, (1, 1, H, W)."""
	image = torch.from_numpy(np.array(Image.open(path).convert("L"))).float()
	image = image / torch.max(image)
	image = torch.movedim(image, -1, 0).unsqueeze(0).unsqueeze(0)
	return image


def get_coords(H: int, W: int) -> torch.Tensor:
	"""Creates a coordinates grid for INF."""
	coords = np.dstack(np.meshgrid(np.linspace(0, 1, H), np.linspace(0, 1, W)))
	coords = torch.from_numpy(coords).float().cuda()
	return coords


def ff_embedding(image: torch.Tensor, B: torch.Tensor = None) -> torch.Tensor:
	if B is None:
		return image
	else:
		x_proj    = (2. * np.pi * image) @ B.T
		embedding = torch.cat([torch.sin(x_proj), torch.cos(x_proj)], axis=-1)
		return embedding


def get_patches(image: torch.Tensor, kernel_size: int = 1) -> torch.Tensor:
	"""Creates a tensor where the channel contains patch information."""
	kernel = torch.zeros((kernel_size ** 2, 1, kernel_size, kernel_size)).cuda()

	for i in range(kernel_size):
		for j in range(kernel_size):
			kernel[int(torch.sum(kernel).item()), 0, i, j] = 1

	pad 	  = nn.ReflectionPad2d(kernel_size // 2)
	im_padded = pad(image)
	extracted = torch.nn.functional.conv2d(im_padded, kernel, padding=0).squeeze(0)
	return torch.movedim(extracted, 0, -1)


def get_mgrid(side_length: int, dim: int = 2):
	"""Generates a flattened grid of (x,y,...) coordinates in a range of -1 to 1.
    sidelen: int
    dim: int
    """
	tensors = tuple(dim * [torch.linspace(-1, 1, steps=side_length)])
	mgrid   = torch.stack(torch.meshgrid(*tensors), dim=-1)
	mgrid   = mgrid.reshape(-1, dim)
	return mgrid


def interpolate_image(image: torch.Tensor, H: int, W: int) -> torch.Tensor:
	"""Reshapes the image based on new resolution."""
	return F.interpolate(image, size=(H, W))


def get_v_component(image_hsv: torch.Tensor) -> torch.Tensor:
	"""Assumes (1, 3, H, W) HSV image."""
	return image_hsv[:, -1].unsqueeze(0)


def replace_v_component(image_hsv: torch.Tensor, v_new: torch.Tensor) -> torch.Tensor:
	"""Replaces the V component of a HSV image (1, 3, H, W)."""
	image_hsv[:, -1] = v_new
	return image_hsv


def rgb2hsv_torch(rgb: torch.Tensor) -> torch.Tensor:
	cmax, cmax_idx = torch.max(rgb, dim=1, keepdim=True)
	cmin  = torch.min(rgb, dim=1, keepdim=True)[0]
	delta = cmax - cmin
	hsv_h = torch.empty_like(rgb[:, 0:1, :, :])
	cmax_idx[delta == 0] = 3
	hsv_h[cmax_idx == 0] = (((rgb[:, 1:2] - rgb[:, 2:3]) / delta) % 6)[cmax_idx == 0]
	hsv_h[cmax_idx == 1] = (((rgb[:, 2:3] - rgb[:, 0:1]) / delta) + 2)[cmax_idx == 1]
	hsv_h[cmax_idx == 2] = (((rgb[:, 0:1] - rgb[:, 1:2]) / delta) + 4)[cmax_idx == 2]
	hsv_h[cmax_idx == 3] = 0.0
	hsv_h /= 6.0
	hsv_s = torch.where(cmax == 0, torch.tensor(0.).type_as(rgb), delta / cmax)
	hsv_v = cmax
	return torch.cat([hsv_h, hsv_s, hsv_v], dim=1)


def hsv2rgb_torch(hsv: torch.Tensor) -> torch.Tensor:
	hsv_h, hsv_s, hsv_l = hsv[:, 0:1], hsv[:, 1:2], hsv[:, 2:3]
	_c  = hsv_l * hsv_s
	_x  = _c * (- torch.abs(hsv_h * 6.0 % 2.0 - 1) + 1.0)
	_m  = hsv_l - _c
	_o  = torch.zeros_like(_c)
	idx = (hsv_h * 6.0).type(torch.uint8)
	idx = (idx % 6).expand(-1, 3, -1, -1)
	rgb = torch.empty_like(hsv)
	rgb[idx == 0] = torch.cat([_c, _x, _o], dim=1)[idx == 0]
	rgb[idx == 1] = torch.cat([_x, _c, _o], dim=1)[idx == 1]
	rgb[idx == 2] = torch.cat([_o, _c, _x], dim=1)[idx == 2]
	rgb[idx == 3] = torch.cat([_o, _x, _c], dim=1)[idx == 3]
	rgb[idx == 4] = torch.cat([_x, _o, _c], dim=1)[idx == 4]
	rgb[idx == 5] = torch.cat([_c, _o, _x], dim=1)[idx == 5]
	rgb += _m
	return rgb


def diff_x(input: torch.Tensor, r: int) -> torch.Tensor:
	assert input.dim() == 4
	left   = input[:, :,         r:2 * r + 1]
	middle = input[:, :, 2 * r + 1:         ] - input[:, :,           :-2 * r - 1]
	right  = input[:, :,        -1:         ] - input[:, :, -2 * r - 1:    -r - 1]
	output = torch.cat([left, middle, right], dim=2)
	return output


def diff_y(input: torch.Tensor, r: int) -> torch.Tensor:
	assert input.dim() == 4
	left   = input[:, :, :,         r:2 * r + 1]
	middle = input[:, :, :, 2 * r + 1:         ] - input[:, :, :,           :-2 * r - 1]
	right  = input[:, :, :,        -1:         ] - input[:, :, :, -2 * r - 1:    -r - 1]
	output = torch.cat([left, middle, right], dim=3)
	return output


class BoxFilter(nn.Module):

	def __init__(self, r: int):
		super().__init__()
		self.r = r

	def forward(self, x):
		assert x.dim() == 4
		return diff_y(diff_x(x.cumsum(dim=2), self.r).cumsum(dim=3), self.r)


class FastGuidedFilter(nn.Module):

	def __init__(self, r: int, eps: float =1e-8):
		super().__init__()
		self.r		   = r
		self.eps 	   = eps
		self.boxfilter = BoxFilter(r)


	def forward(self, lr_x, lr_y, hr_x):
		n_lrx, c_lrx, h_lrx, w_lrx = lr_x.size()
		n_lry, c_lry, h_lry, w_lry = lr_y.size()
		n_hrx, c_hrx, h_hrx, w_hrx = hr_x.size()

		assert n_lrx == n_lry and n_lry == n_hrx
		assert c_lrx == c_hrx and (c_lrx == 1 or c_lrx == c_lry)
		assert h_lrx == h_lry and w_lrx == w_lry
		assert h_lrx > 2*self.r+1 and w_lrx > 2*self.r+1

		## N
		N = self.boxfilter(Variable(lr_x.data.new().resize_((1, 1, h_lrx, w_lrx)).fill_(1.0)))

		## mean_x
		mean_x = self.boxfilter(lr_x) / N
		## mean_y
		mean_y = self.boxfilter(lr_y) / N
		## cov_xy
		cov_xy = self.boxfilter(lr_x * lr_y) / N - mean_x * mean_y
		## var_x
		var_x = self.boxfilter(lr_x * lr_x) / N - mean_x * mean_x

		## A
		A = cov_xy / (var_x + self.eps)
		## b
		b = mean_y - A * mean_x

		## mean_A; mean_b
		mean_A = F.interpolate(A, (h_hrx, w_hrx), mode='bilinear', align_corners=True)
		mean_b = F.interpolate(b, (h_hrx, w_hrx), mode='bilinear', align_corners=True)

		return mean_A*hr_x+mean_b


def filter_up(x_lr: torch.Tensor, y_lr: torch.Tensor, x_hr: torch.Tensor, r: int = 1) -> torch.Tensor:
	"""Applies the guided filter to upscale the predicted image."""
	guided_filter = FastGuidedFilter(r=r)
	y_hr = guided_filter(x_lr, y_lr, x_hr)
	y_hr = torch.clip(y_hr, 0, 1)
	return y_hr

## Model

### Activation Layers

In [374]:
class SigmoidLayer(nn.Module):

    def __init__(self, in_channels: int, out_channels: int, *args, **kwargs):
        super().__init__()
        self.in_channels = in_channels
        self.linear      = nn.Linear(in_channels, out_channels)
        self.act         = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.act(self.linear(x))
        # return self.linear(x)

In [375]:
class TanhLayer(nn.Module):

    def __init__(self, in_channels: int, out_channels: int, *args, **kwargs):
        super().__init__()
        self.in_channels = in_channels
        self.linear      = nn.Linear(in_channels, out_channels)
        self.act         = nn.Tanh()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.act(self.linear(x))

In [376]:
class ReLULayer(nn.Module):

	def __init__(self, in_channels: int, out_channels: int, *args, **kwargs):
		super().__init__()
		self.in_channels = in_channels
		self.linear      = nn.Linear(in_channels, out_channels)
		self.act         = nn.ReLU()

	def forward(self, x: torch.Tensor) -> torch.Tensor:
		return self.act(self.linear(x))

In [377]:
class SineLayer(nn.Module):

	def __init__(
		self,
		in_channels : int,
		out_channels: int,
		omega_0 	: float = 30,
		is_first	: bool  = False,
	):
		super().__init__()
		self.in_channels = in_channels
		self.omega_0     = omega_0
		self.linear      = nn.Linear(in_channels, out_channels)
		self.is_first    = is_first
		self.init_weights()

	def init_weights(self):
		b = 1.0 / self.in_channels if self.is_first else np.sqrt(6.0 / self.in_channels) / self.omega_0
		with torch.no_grad():
			self.linear.weight.uniform_(-b, b)

	def forward(self, x: torch.Tensor) -> torch.Tensor:
		return torch.sin(self.omega_0 * self.linear(x))

In [378]:
class GaussLayer(nn.Module):

    def __init__(
        self,
        in_channels : int,
        out_channels: int,
        scale       : float = 10.0,
        *args, **kwargs
    ):
        super().__init__()
        self.in_channels = in_channels
        self.scale       = scale
        self.linear      = nn.Linear(in_channels, out_channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return torch.exp(-(self.scale * self.linear(x)) ** 2)

In [379]:
class FINERLayer(nn.Module):

	def __init__(
		self,
		in_channels     : int,
		out_channels    : int,
		omega_0         : float = 30.0,
		first_bias_scale: float = None,
		is_first        : bool  = False,
		scale_req_grad  : bool  = False,
	):
		super().__init__()
		self.omega_0     = omega_0
		self.is_first    = is_first
		self.in_channels = in_channels
		self.linear      = nn.Linear(in_channels, out_channels)

		self.init_weights()
		self.scale_req_grad   = scale_req_grad
		self.first_bias_scale = first_bias_scale
		if self.first_bias_scale is not None:
			self.init_first_bias()

	def init_weights(self):
		with torch.no_grad():
			if self.is_first:
				self.linear.weight.uniform_(-1 / self.in_channels, 1 / self.in_channels)
			else:
				self.linear.weight.uniform_(-np.sqrt(6 / self.in_channels) / self.omega_0,
										 	 np.sqrt(6 / self.in_channels) / self.omega_0)

	def init_first_bias(self):
		with torch.no_grad():
			if self.is_first:
				self.linear.bias.uniform_(-self.first_bias_scale, self.first_bias_scale)

	def generate_scale(self, x: torch.Tensor) -> torch.Tensor:
		if self.scale_req_grad:
			scale = torch.abs(x) + 1
		else:
			with torch.no_grad():
				scale = torch.abs(x) + 1
		return scale

	def forward(self, x: torch.Tensor) -> torch.Tensor:
		linear = self.linear(x)
		scale  = self.generate_scale(linear)
		return torch.sin(self.omega_0 * scale * linear)

In [380]:
class FINERLayer20(nn.Module):

	def __init__(
		self,
		in_channels     : int,
		out_channels    : int,
		omega_0         : float = 30.0,
		first_bias_scale: float = 20.0,
		is_first        : bool  = False,
		scale_req_grad  : bool  = False,
	):
		super().__init__()
		self.omega_0     = omega_0
		self.is_first    = is_first
		self.in_channels = in_channels
		self.linear      = nn.Linear(in_channels, out_channels)

		self.init_weights()
		self.scale_req_grad   = scale_req_grad
		self.first_bias_scale = first_bias_scale
		if self.first_bias_scale is not None:
			self.init_first_bias()

	def init_weights(self):
		with torch.no_grad():
			if self.is_first:
				self.linear.weight.uniform_(-1 / self.in_channels, 1 / self.in_channels)
			else:
				self.linear.weight.uniform_(-np.sqrt(6 / self.in_channels) / self.omega_0,
											np.sqrt(6 / self.in_channels) / self.omega_0)

	def init_first_bias(self):
		with torch.no_grad():
			if self.is_first:
				self.linear.bias.uniform_(-self.first_bias_scale, self.first_bias_scale)

	def generate_scale(self, x: torch.Tensor) -> torch.Tensor:
		if self.scale_req_grad:
			scale = torch.abs(x) + 1
		else:
			with torch.no_grad():
				scale = torch.abs(x) + 1
		return scale

	def forward(self, x: torch.Tensor) -> torch.Tensor:
		linear = self.linear(x)
		scale  = self.generate_scale(linear)
		return torch.sin(self.omega_0 * scale * linear)

### Define INF module

In [381]:
class INF1_Patch(nn.Module):

	def __init__(
		self,
		patch_dim   : int   = 1 ** 2,
		hidden_dim  : int   = 256,
		num_layers  : int   = 4,
		add_layer   : int   = 2,
		v_act_layer : Any   = SineLayer,
		use_ff      : bool  = False,
		ff_scale	: float = 10,
		weight_decay: float = None
	):
		super().__init__()
		if use_ff:
			self.register_buffer("B", torch.randn((hidden_dim, patch_dim)) * ff_scale)
			patch_in_channels   = hidden_dim * 2
		else:
			self.B 				= None
			patch_in_channels   = patch_dim

		patch_layers   = [v_act_layer(patch_in_channels, hidden_dim, is_first=True)]
		output_layers  = []

		for _ in range(1, add_layer - 2):
			patch_layers.append(v_act_layer(hidden_dim, hidden_dim))
		patch_layers.append(v_act_layer(hidden_dim, hidden_dim))

		for _ in range(add_layer, num_layers - 1):
			output_layers.append(v_act_layer(hidden_dim, hidden_dim))
		output_layers.append(SigmoidLayer(hidden_dim, 1))

		self.patch_net   = nn.Sequential(*patch_layers)
		self.output_net  = nn.Sequential(*output_layers)

		if not weight_decay:
			weight_decay = [0.1, 0.0001, 0.001]

		self.params  = []
		self.params += [{"params": self.patch_net.parameters(),   "weight_decay": weight_decay[1]}]
		self.params += [{"params": self.output_net.parameters(),  "weight_decay": weight_decay[2]}]

	def forward(self, spatial: torch.Tensor, patch: torch.Tensor) -> torch.Tensor:
		patch = ff_embedding(patch, self.B)
		return self.output_net(self.patch_net(patch))

In [382]:
class INF1_Spatial(nn.Module):

	def __init__(
		self,
		patch_dim   : int   = 1 ** 2,
		hidden_dim  : int   = 256,
		num_layers  : int   = 4,
		add_layer   : int   = 2,
		s_act_layer : Any   = SineLayer,
		use_ff      : bool  = False,
		ff_scale	: float = 10,
		weight_decay: float = None
	):
		super().__init__()
		if use_ff:
			self.register_buffer("B", torch.randn((hidden_dim, 2)) * ff_scale)
			spatial_in_channels = hidden_dim * 2
		else:
			self.B 				= None
			spatial_in_channels = 2

		spatial_layers = [s_act_layer(spatial_in_channels, hidden_dim, is_first=True)]
		output_layers  = []

		for _ in range(1, add_layer - 2):
			spatial_layers.append(s_act_layer(hidden_dim, hidden_dim))
		spatial_layers.append(s_act_layer(hidden_dim, hidden_dim))

		for _ in range(add_layer, num_layers - 1):
			output_layers.append(s_act_layer(hidden_dim, hidden_dim))
		output_layers.append(SigmoidLayer(hidden_dim, 1))

		self.spatial_net = nn.Sequential(*spatial_layers)
		self.output_net  = nn.Sequential(*output_layers)

		if not weight_decay:
			weight_decay = [0.1, 0.0001, 0.001]

		self.params  = []
		self.params += [{"params": self.spatial_net.parameters(), "weight_decay": weight_decay[0]}]
		self.params += [{"params": self.output_net.parameters(),  "weight_decay": weight_decay[2]}]

	def forward(self, spatial: torch.Tensor, patch: torch.Tensor) -> torch.Tensor:
		spatial = ff_embedding(spatial, self.B)
		return self.output_net(self.spatial_net(spatial))

In [383]:
class INF2(nn.Module):

	def __init__(
		self,
		patch_dim   : int   = 1 ** 2,
		hidden_dim  : int   = 256,
		num_layers  : int   = 4,
		add_layer   : int   = 2,
		v_act_layer : Any   = SineLayer,
		s_act_layer : Any   = SineLayer,
		use_ff      : bool  = False,
		ff_scale	: float = 10,
		weight_decay: float = None
	):
		super().__init__()
		if use_ff:
			self.register_buffer("B1", torch.randn((hidden_dim, 2)) * ff_scale)
			spatial_in_channels = hidden_dim * 2
			self.register_buffer("B2", torch.randn((hidden_dim, patch_dim)) * ff_scale)
			patch_in_channels   = hidden_dim * 2
		else:
			self.B1 		    = None
			self.B2 		    = None
			spatial_in_channels = 2
			patch_in_channels   = patch_dim

		spatial_layers = [s_act_layer(spatial_in_channels, hidden_dim, is_first=True)]
		patch_layers   = [v_act_layer(patch_in_channels,   hidden_dim, is_first=True)]
		output_layers  = []

		for _ in range(1, add_layer - 2):
			spatial_layers.append(s_act_layer(hidden_dim, hidden_dim))
			patch_layers.append(  v_act_layer(hidden_dim, hidden_dim))
		spatial_layers.append(s_act_layer(hidden_dim, hidden_dim))
		patch_layers.append(  v_act_layer(hidden_dim, hidden_dim))

		output_layers.append(v_act_layer(hidden_dim * 2, hidden_dim))
		for _ in range(add_layer + 1, num_layers - 1):
			output_layers.append(v_act_layer(hidden_dim, hidden_dim))
		output_layers.append(SigmoidLayer(hidden_dim, 1))

		self.patch_net   = nn.Sequential(*patch_layers)
		self.spatial_net = nn.Sequential(*spatial_layers)
		self.output_net  = nn.Sequential(*output_layers)

		if not weight_decay:
			weight_decay = [0.1, 0.0001, 0.001]

		self.params  = []
		self.params += [{"params": self.spatial_net.parameters(), "weight_decay": weight_decay[0]}]
		self.params += [{"params": self.patch_net.parameters(),   "weight_decay": weight_decay[1]}]
		self.params += [{"params": self.output_net.parameters(),  "weight_decay": weight_decay[2]}]

	def forward(self, spatial: torch.Tensor, patch: torch.Tensor) -> torch.Tensor:
		spatial = ff_embedding(spatial, self.B1)
		patch   = ff_embedding(patch,   self.B2)
		return self.output_net(torch.cat((self.spatial_net(spatial), self.patch_net(patch)), -1))

In [384]:
class INF4(nn.Module):

	def __init__(
		self,
		patch_dim   : int   = 1 ** 2,
		hidden_dim  : int   = 256,
		num_layers  : int   = 4,
		add_layer   : int   = 2,
		v_act_layer : Any   = SineLayer,
		s_act_layer : Any   = SineLayer,
		use_ff      : bool  = False,
		ff_scale	: float = 10,
		weight_decay: float = None
	):
		super().__init__()
		if use_ff:
			self.register_buffer("B1", torch.randn((hidden_dim, 2)) * ff_scale)
			spatial_in_channels = hidden_dim * 2
			self.register_buffer("B2", torch.randn((hidden_dim, patch_dim)) * ff_scale)
			patch_in_channels   = hidden_dim * 2
		else:
			self.B1 		    = None
			self.B2 		    = None
			spatial_in_channels = 2
			patch_in_channels   = patch_dim

		spatial_layers = [s_act_layer(spatial_in_channels, hidden_dim, is_first=True)]
		patch_layers1  = [v_act_layer(patch_in_channels,   hidden_dim, is_first=True)]
		patch_layers2  = [v_act_layer(patch_in_channels,   hidden_dim, is_first=True)]
		patch_layers3  = [v_act_layer(patch_in_channels,   hidden_dim, is_first=True)]
		output_layers  = []

		for _ in range(1, add_layer - 2):
			spatial_layers.append(s_act_layer(hidden_dim, hidden_dim))
			patch_layers1.append( v_act_layer(hidden_dim, hidden_dim))
			patch_layers2.append( v_act_layer(hidden_dim, hidden_dim))
			patch_layers3.append( v_act_layer(hidden_dim, hidden_dim))
		spatial_layers.append(s_act_layer(hidden_dim, hidden_dim))
		patch_layers1.append( v_act_layer(hidden_dim, hidden_dim))
		patch_layers2.append( v_act_layer(hidden_dim, hidden_dim))
		patch_layers3.append( v_act_layer(hidden_dim, hidden_dim))

		output_layers.append(v_act_layer(hidden_dim * 4, hidden_dim))
		for _ in range(add_layer + 1, num_layers - 1):
			output_layers.append(v_act_layer(hidden_dim, hidden_dim))
		output_layers.append(SigmoidLayer(hidden_dim, 1))

		self.patch_net1  = nn.Sequential(*patch_layers1)
		self.patch_net2  = nn.Sequential(*patch_layers2)
		self.patch_net3  = nn.Sequential(*patch_layers3)
		self.spatial_net = nn.Sequential(*spatial_layers)
		self.output_net  = nn.Sequential(*output_layers)

		if not weight_decay:
			weight_decay = [0.1, 0.0001, 0.001]

		self.params  = []
		self.params += [{"params": self.spatial_net.parameters(), "weight_decay": weight_decay[0]}]
		self.params += [{"params": self.patch_net1.parameters(),  "weight_decay": weight_decay[1]}]
		self.params += [{"params": self.patch_net2.parameters(),  "weight_decay": weight_decay[1]}]
		self.params += [{"params": self.patch_net3.parameters(),  "weight_decay": weight_decay[1]}]
		self.params += [{"params": self.output_net.parameters(),  "weight_decay": weight_decay[2]}]

	def forward(self, spatial: torch.Tensor, patch1: torch.Tensor, patch2: torch.Tensor, patch3: torch.Tensor) -> torch.Tensor:
		spatial = ff_embedding(spatial, self.B1)
		patch1  = ff_embedding(patch1,  self.B2)
		patch2  = ff_embedding(patch2,  self.B2)
		patch3  = ff_embedding(patch3,  self.B2)
		return self.output_net(torch.cat((self.patch_net1(patch1), self.patch_net2(patch2), self.patch_net3(patch3), self.spatial_net(spatial)), -1))

## Training Loop

In [385]:
class L_exp(nn.Module):

	def __init__(self, patch_size, mean_val):
		super().__init__()
		self.pool     = nn.AvgPool2d(patch_size)
		self.mean_val = mean_val

	def forward(self, x):
		mean = self.pool(x) ** 0.5
		d 	 = torch.abs(torch.mean(torch.pow(mean - torch.FloatTensor([self.mean_val] ).cuda(), 2)))
		return d


class L_TV(nn.Module):

	def __init__(self):
		super().__init__()

	def forward(self, x):
		batch_size = x.size()[0]
		h_x 	   = x.size()[2]
		w_x 	   = x.size()[3]
		count_h    = (x.size()[2] - 1) * x.size()[3]
		count_w    = x.size()[2] * (x.size()[3] - 1)
		h_tv 	   = torch.pow((x[:,:,1:,:] - x[:,:,:h_x-1,:]), 2).sum()
		w_tv 	   = torch.pow((x[:,:,:,1:] - x[:,:,:,:w_x-1]), 2).sum()
		return 2 * (h_tv / count_h + w_tv / count_w) / batch_size

In [386]:
def calculate_psnr(img1: torch.Tensor, img2: torch.Tensor, max_pixel_value: float = 1.0) -> float:
	"""Calculate the Peak Signal-to-Noise Ratio (PSNR) between two images."""
	assert img1.shape == img2.shape, "Input images must have the same dimensions"
	mse  = F.mse_loss(img1, img2, reduction='mean').item()
	if mse == 0:
		return float("inf")  # Identical images
	psnr = 20 * torch.log10(torch.tensor(max_pixel_value)) - 10 * torch.log10(torch.tensor(mse))
	return psnr.item()

In [387]:
def train(model, total_steps, steps_til_summary, input_type="pvde"):
	model.to(device)
	optimizer = torch.optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.999), weight_decay=3e-4)
	l_exp 	  = L_exp(16, L)
	l_tv 	  = L_TV()

    # Input
	image, depth, ref = get_image_tensor()
	image, depth, ref = image.to(device), depth.to(device), ref.to(device)

	coords    = get_coords(down_size, down_size)

	global image_v
	image_hsv = rgb2hsv_torch(image)
	image_v   = get_v_component(image_hsv)
	image_v   = interpolate_image(image_v, down_size, down_size)

	edge      = mon.BoundaryAwarePrior(0.05)(depth)
	depth     = interpolate_image(depth, down_size, down_size)
	edge      = interpolate_image(edge,  down_size, down_size)

	global ref_v
	ref_hsv   = rgb2hsv_torch(ref)
	ref_v     = get_v_component(ref_hsv)
	ref_v     = interpolate_image(ref_v, down_size, down_size)

	# Groundtruth Residual
	global res_gt
	illu_gt   = image_v / ref_v
	res_gt    = illu_gt - image_v

	if input_type == "pv":
		patches  = get_patches(image_v, window_size)
	elif input_type == "pd":
		patches  = get_patches(depth, window_size)
	elif input_type == "pe":
		patches  = get_patches(depth, window_size)
	elif input_type == "pvde":
		patches1 = get_patches(image_v, window_size)
		patches2 = get_patches(depth,   window_size)
		patches3 = get_patches(edge,    window_size)
	else:
		patches  = get_patches(image_v, window_size)

    #
	losses  = []
	psnrs   = []
	outputs = []
	refs    = []
	ress    = []
	for step in range(total_steps):
		model.train()
		optimizer.zero_grad()

		if input_type == "pvde":
			illu_res  = model(coords, patches1, patches2, patches3)
		else:
			illu_res  = model(coords, patches)
		illu_res      = illu_res.view(1, 1, down_size, down_size)
		illu          = illu_res + image_v
		image_v_fixed = image_v / (illu + 1e-4)

		loss_spa 	  = torch.mean(torch.abs(torch.pow(illu - image_v, 2)))
		loss_tv  	  = l_tv(illu)
		loss_exp 	  = torch.mean(l_exp(illu))
		loss_sparsity = torch.mean(image_v_fixed)
		loss          = torch.mean(torch.abs(torch.pow(image_v_fixed - ref_v, 2)))
		# loss 		  = loss_spa * 1 + loss_tv * 20 + loss_exp * 9 + loss_sparsity * 5
		psnr 		  = calculate_psnr(image_v_fixed, ref_v)

		losses.append(loss.item())
		psnrs.append(psnr)
		if not step % steps_til_summary or (step == total_steps - 1):
			print("Step %d, Loss %0.6f, PSNR %0.6f" % (step, loss, psnr))
			outputs.append(image_v_fixed.cpu().view(down_size, down_size).detach().numpy())
			refs.append(ref_v.cpu().view(down_size, down_size).detach().numpy())
			ress.append(illu_res.squeeze(0).squeeze(0).cpu().detach().numpy())
		loss.backward()
		optimizer.step()

	# image_v_fixed   = filter_up(image_v_lr, image_v_fixed_lr, image_v)
	# image_hsv_fixed = replace_v_component(image_hsv, image_v_fixed)
	# image_rgb_fixed = hsv2rgb_torch(image_hsv_fixed)
	# image_rgb_fixed = image_rgb_fixed / torch.max(image_rgb_fixed)

	return losses, psnrs, outputs, ress, refs

## Define Models

In [388]:
siren_temp = INF1_Spatial(patch_dim=window_size**2, s_act_layer=SineLayer)  # f: p -> r

### ReLU

In [389]:
relu_p  = INF1_Spatial(patch_dim=window_size**2, s_act_layer=ReLULayer)  # f: p -> r
relu_v  =   INF1_Patch(patch_dim=window_size**2, v_act_layer=ReLULayer)  # f: v -> r
relu_d  =   INF1_Patch(patch_dim=window_size**2, v_act_layer=ReLULayer)  # f: d -> r
relu_e  =   INF1_Patch(patch_dim=window_size**2, v_act_layer=ReLULayer)  # f: e -> r
relu_pv = 	      INF2(patch_dim=window_size**2, v_act_layer=ReLULayer, s_act_layer=ReLULayer)  # f: pv -> r
relu_pd = 	      INF2(patch_dim=window_size**2, v_act_layer=ReLULayer, s_act_layer=ReLULayer)  # f: pd -> r
relu_pe =         INF2(patch_dim=window_size**2, v_act_layer=ReLULayer, s_act_layer=ReLULayer)  # f: pe -> r
relu    =         INF4(patch_dim=window_size**2, v_act_layer=ReLULayer, s_act_layer=ReLULayer)  # f: pvde -> r

ff_p    = INF1_Spatial(patch_dim=window_size**2, s_act_layer=ReLULayer, use_ff=True)
ff_v    =   INF1_Patch(patch_dim=window_size**2, v_act_layer=ReLULayer, use_ff=True)
ff_d    =   INF1_Patch(patch_dim=window_size**2, v_act_layer=ReLULayer, use_ff=True)
ff_e    =   INF1_Patch(patch_dim=window_size**2, v_act_layer=ReLULayer, use_ff=True)
ff_pv   =         INF2(patch_dim=window_size**2, v_act_layer=ReLULayer, s_act_layer=ReLULayer, use_ff=True)
ff_pd   =         INF2(patch_dim=window_size**2, v_act_layer=ReLULayer, s_act_layer=ReLULayer, use_ff=True)
ff_pe   =         INF2(patch_dim=window_size**2, v_act_layer=ReLULayer, s_act_layer=ReLULayer, use_ff=True)
ff      =         INF4(patch_dim=window_size**2, v_act_layer=ReLULayer, s_act_layer=ReLULayer, use_ff=True)

### Gauss

In [390]:
gauss_p     = INF1_Spatial(patch_dim=window_size**2, s_act_layer=GaussLayer)  # f: p -> r
gauss_v     =   INF1_Patch(patch_dim=window_size**2, v_act_layer=GaussLayer)  # f: v -> r
gauss_d     =   INF1_Patch(patch_dim=window_size**2, v_act_layer=GaussLayer)  # f: d -> r
gauss_e     =   INF1_Patch(patch_dim=window_size**2, v_act_layer=GaussLayer)  # f: e -> r
gauss_pv    = 	      INF2(patch_dim=window_size**2, v_act_layer=GaussLayer, s_act_layer=GaussLayer)  # f: pv -> r
gauss_pd    = 	      INF2(patch_dim=window_size**2, v_act_layer=GaussLayer, s_act_layer=GaussLayer)  # f: pd -> r
gauss_pe    =         INF2(patch_dim=window_size**2, v_act_layer=GaussLayer, s_act_layer=GaussLayer)  # f: pe -> r
gauss       =         INF4(patch_dim=window_size**2, v_act_layer=GaussLayer, s_act_layer=GaussLayer)  # f: pvde -> r

ff_gauss_p  = INF1_Spatial(patch_dim=window_size**2, s_act_layer=GaussLayer, use_ff=True)
ff_gauss_v  =   INF1_Patch(patch_dim=window_size**2, v_act_layer=GaussLayer, use_ff=True)
ff_gauss_d  =   INF1_Patch(patch_dim=window_size**2, v_act_layer=GaussLayer, use_ff=True)
ff_gauss_e  =   INF1_Patch(patch_dim=window_size**2, v_act_layer=GaussLayer, use_ff=True)
ff_gauss_pv = 		  INF2(patch_dim=window_size**2, v_act_layer=GaussLayer, s_act_layer=GaussLayer, use_ff=True)
ff_gauss_pd = 		  INF2(patch_dim=window_size**2, v_act_layer=GaussLayer, s_act_layer=GaussLayer, use_ff=True)
ff_gauss_pe =         INF2(patch_dim=window_size**2, v_act_layer=GaussLayer, s_act_layer=GaussLayer, use_ff=True)
ff_gauss    =         INF4(patch_dim=window_size**2, v_act_layer=GaussLayer, s_act_layer=GaussLayer, use_ff=True)

### SIREN

In [391]:
siren_p     = INF1_Spatial(patch_dim=window_size**2, s_act_layer=SineLayer)  # f: p -> r
siren_v     =   INF1_Patch(patch_dim=window_size**2, v_act_layer=SineLayer)  # f: v -> r
siren_d     =   INF1_Patch(patch_dim=window_size**2, v_act_layer=SineLayer)  # f: d -> r
siren_e     =   INF1_Patch(patch_dim=window_size**2, v_act_layer=SineLayer)  # f: e -> r
siren_pv    = 		  INF2(patch_dim=window_size**2, v_act_layer=SineLayer, s_act_layer=SineLayer)  # f: pv -> r
siren_pd    = 		  INF2(patch_dim=window_size**2, v_act_layer=SineLayer, s_act_layer=SineLayer)  # f: pd -> r
siren_pe    =         INF2(patch_dim=window_size**2, v_act_layer=SineLayer, s_act_layer=SineLayer)  # f: pe -> r
siren       =         INF4(patch_dim=window_size**2, v_act_layer=SineLayer, s_act_layer=SineLayer)  # f: pvde -> r

ff_siren_p  = INF1_Spatial(patch_dim=window_size**2, s_act_layer=SineLayer, use_ff=True)
ff_siren_v  =   INF1_Patch(patch_dim=window_size**2, v_act_layer=SineLayer, use_ff=True)
ff_siren_d  =   INF1_Patch(patch_dim=window_size**2, v_act_layer=SineLayer, use_ff=True)
ff_siren_e  =   INF1_Patch(patch_dim=window_size**2, v_act_layer=SineLayer, use_ff=True)
ff_siren_pv = 		  INF2(patch_dim=window_size**2, v_act_layer=SineLayer, s_act_layer=SineLayer, use_ff=True)
ff_siren_pd = 		  INF2(patch_dim=window_size**2, v_act_layer=SineLayer, s_act_layer=SineLayer, use_ff=True)
ff_siren_pe =         INF2(patch_dim=window_size**2, v_act_layer=SineLayer, s_act_layer=SineLayer, use_ff=True)
ff_siren    =         INF4(patch_dim=window_size**2, v_act_layer=SineLayer, s_act_layer=SineLayer, use_ff=True)

# print(siren_pv)
# print(sum(p.numel() for p in siren_pv.parameters()) / 1e6, "M parameters")
# print(f"total pixels {down_size * down_size / 1e6}")

### FINER

In [392]:
finer_p     = INF1_Spatial(patch_dim=window_size**2, s_act_layer=FINERLayer)  # f: p -> r
finer_v     =   INF1_Patch(patch_dim=window_size**2, v_act_layer=FINERLayer)  # f: v -> r
finer_d     =   INF1_Patch(patch_dim=window_size**2, v_act_layer=FINERLayer)  # f: d -> r
finer_e     =   INF1_Patch(patch_dim=window_size**2, v_act_layer=FINERLayer)  # f: e -> r
finer_pv    = 	  	  INF2(patch_dim=window_size**2, v_act_layer=FINERLayer, s_act_layer=FINERLayer)  # f: pv -> r
finer_pd    = 		  INF2(patch_dim=window_size**2, v_act_layer=FINERLayer, s_act_layer=FINERLayer)  # f: pd -> r
finer_pe    =         INF2(patch_dim=window_size**2, v_act_layer=FINERLayer, s_act_layer=FINERLayer)  # f: pe -> r
finer       =         INF4(patch_dim=window_size**2, v_act_layer=FINERLayer, s_act_layer=FINERLayer)  # f: pvde -> r

ff_finer_p  = INF1_Spatial(patch_dim=window_size**2, s_act_layer=FINERLayer, use_ff=True)
ff_finer_v  =   INF1_Patch(patch_dim=window_size**2, v_act_layer=FINERLayer, use_ff=True)
ff_finer_d  =   INF1_Patch(patch_dim=window_size**2, v_act_layer=FINERLayer, use_ff=True)
ff_finer_e  =   INF1_Patch(patch_dim=window_size**2, v_act_layer=FINERLayer, use_ff=True)
ff_finer_pv = 		  INF2(patch_dim=window_size**2, v_act_layer=FINERLayer, s_act_layer=FINERLayer, use_ff=True)
ff_finer_pd = 	      INF2(patch_dim=window_size**2, v_act_layer=FINERLayer, s_act_layer=FINERLayer, use_ff=True)
ff_finer_pe =         INF2(patch_dim=window_size**2, v_act_layer=FINERLayer, s_act_layer=FINERLayer, use_ff=True)
ff_finer    =         INF4(patch_dim=window_size**2, v_act_layer=FINERLayer, s_act_layer=FINERLayer, use_ff=True)

### FINER20

In [393]:
finer20_p     = INF1_Spatial(patch_dim=window_size**2, s_act_layer=FINERLayer20)  # f: p -> r
finer20_v     =   INF1_Patch(patch_dim=window_size**2, v_act_layer=FINERLayer20)  # f: v -> r
finer20_d     =   INF1_Patch(patch_dim=window_size**2, v_act_layer=FINERLayer20)  # f: d -> r
finer20_e     =   INF1_Patch(patch_dim=window_size**2, v_act_layer=FINERLayer20)  # f: e -> r
finer20_pv    = 		INF2(patch_dim=window_size**2, v_act_layer=FINERLayer20, s_act_layer=FINERLayer20)  # f: pv -> r
finer20_pd    = 		INF2(patch_dim=window_size**2, v_act_layer=FINERLayer20, s_act_layer=FINERLayer20)  # f: pd -> r
finer20_pe    =         INF2(patch_dim=window_size**2, v_act_layer=FINERLayer20, s_act_layer=FINERLayer20)  # f: pe -> r
finer20       =         INF4(patch_dim=window_size**2, v_act_layer=FINERLayer20, s_act_layer=FINERLayer20)  # f: pvde -> r

ff_finer20_p  = INF1_Spatial(patch_dim=window_size**2, s_act_layer=FINERLayer20, use_ff=True)
ff_finer20_v  =   INF1_Patch(patch_dim=window_size**2, v_act_layer=FINERLayer20, use_ff=True)
ff_finer20_d  =   INF1_Patch(patch_dim=window_size**2, v_act_layer=FINERLayer20, use_ff=True)
ff_finer20_e  =   INF1_Patch(patch_dim=window_size**2, v_act_layer=FINERLayer20, use_ff=True)
ff_finer20_pv = 		INF2(patch_dim=window_size**2, v_act_layer=FINERLayer20, s_act_layer=FINERLayer20, use_ff=True)
ff_finer20_pd = 	    INF2(patch_dim=window_size**2, v_act_layer=FINERLayer20, s_act_layer=FINERLayer20, use_ff=True)
ff_finer20_pe =         INF2(patch_dim=window_size**2, v_act_layer=FINERLayer20, s_act_layer=FINERLayer20, use_ff=True)
ff_finer20    =         INF4(patch_dim=window_size**2, v_act_layer=FINERLayer20, s_act_layer=FINERLayer20, use_ff=True)

# print(sum(p.numel() for p in siren_pv.parameters()) / 1e6, "M parameters")
# print(f"total pixels {down_size * down_size / 1e6}")

### FIREN

In [394]:
firen_p     = INF1_Spatial(patch_dim=window_size**2, s_act_layer=FINERLayer20)  # f: p -> r
firen_v     =   INF1_Patch(patch_dim=window_size**2, v_act_layer=FINERLayer)  # f: v -> r
firen_d     =   INF1_Patch(patch_dim=window_size**2, v_act_layer=FINERLayer)  # f: d -> r
firen_e     =   INF1_Patch(patch_dim=window_size**2, v_act_layer=FINERLayer)  # f: e -> r
firen_pv    =     	  INF2(patch_dim=window_size**2, v_act_layer=FINERLayer, s_act_layer=FINERLayer20, use_ff=False)
firen_pd    = 	  	  INF2(patch_dim=window_size**2, v_act_layer=FINERLayer, s_act_layer=FINERLayer20, use_ff=False)
firen_pe    = 		  INF2(patch_dim=window_size**2, v_act_layer=FINERLayer, s_act_layer=FINERLayer20, use_ff=False)
firen       = 		  INF4(patch_dim=window_size**2, v_act_layer=FINERLayer, s_act_layer=FINERLayer20, use_ff=False)

ff_firen_pv = 		  INF2(patch_dim=window_size**2, v_act_layer=FINERLayer, s_act_layer=FINERLayer20, use_ff=True)
ff_firen_pd = 		  INF2(patch_dim=window_size**2, v_act_layer=FINERLayer, s_act_layer=FINERLayer20, use_ff=True)
ff_firen_pe =		  INF2(patch_dim=window_size**2, v_act_layer=FINERLayer, s_act_layer=FINERLayer20, use_ff=True)
ff_firen    =		  INF4(patch_dim=window_size**2, v_act_layer=FINERLayer, s_act_layer=FINERLayer20, use_ff=True)  # f: pvde -> r

## Begin Training

In [395]:
results = train(siren_temp, total_steps, summary_steps, input_type="pv")
outputs["ref"] = results[4]

Step 0, Loss 0.014487, PSNR 18.390285
Step 10, Loss 0.004612, PSNR 23.361332
Step 20, Loss 0.003524, PSNR 24.529135
Step 30, Loss 0.003056, PSNR 25.148411
Step 40, Loss 0.002903, PSNR 25.372128
Step 50, Loss 0.002824, PSNR 25.490681
Step 60, Loss 0.002798, PSNR 25.532076
Step 70, Loss 0.002783, PSNR 25.555649
Step 80, Loss 0.002774, PSNR 25.568924
Step 90, Loss 0.002767, PSNR 25.579838
Step 100, Loss 0.002761, PSNR 25.589373
Step 110, Loss 0.002756, PSNR 25.597525
Step 120, Loss 0.002751, PSNR 25.604778
Step 130, Loss 0.002747, PSNR 25.611408
Step 140, Loss 0.002743, PSNR 25.617487
Step 150, Loss 0.002740, PSNR 25.623127
Step 160, Loss 0.002736, PSNR 25.628378
Step 170, Loss 0.002733, PSNR 25.633295
Step 180, Loss 0.002730, PSNR 25.637913
Step 190, Loss 0.002728, PSNR 25.642265
Step 200, Loss 0.002725, PSNR 25.646383
Step 210, Loss 0.002723, PSNR 25.650284
Step 220, Loss 0.002720, PSNR 25.653992
Step 230, Loss 0.002718, PSNR 25.657520
Step 240, Loss 0.002716, PSNR 25.660892
Step 250, L

### ReLU

In [396]:
losses["relu_p"], psnrs["relu_p"], outputs["relu_p"], ress["relu_p"], _ = train(relu_p, total_steps, summary_steps, input_type="pv")

Step 0, Loss 0.012022, PSNR 19.200384
Step 10, Loss 0.011648, PSNR 19.337631
Step 20, Loss 0.011275, PSNR 19.478930
Step 30, Loss 0.010906, PSNR 19.623425
Step 40, Loss 0.010540, PSNR 19.771484
Step 50, Loss 0.010175, PSNR 19.924805
Step 60, Loss 0.009806, PSNR 20.085173
Step 70, Loss 0.009432, PSNR 20.253965
Step 80, Loss 0.009052, PSNR 20.432476
Step 90, Loss 0.008664, PSNR 20.622631
Step 100, Loss 0.008272, PSNR 20.823719
Step 110, Loss 0.007879, PSNR 21.035168
Step 120, Loss 0.007485, PSNR 21.258089
Step 130, Loss 0.007095, PSNR 21.490673
Step 140, Loss 0.006712, PSNR 21.731550
Step 150, Loss 0.006340, PSNR 21.979446
Step 160, Loss 0.005980, PSNR 22.232950
Step 170, Loss 0.005636, PSNR 22.490379
Step 180, Loss 0.005310, PSNR 22.749199
Step 190, Loss 0.005004, PSNR 23.006876
Step 200, Loss 0.004720, PSNR 23.260614
Step 210, Loss 0.004460, PSNR 23.506308
Step 220, Loss 0.004227, PSNR 23.739239
Step 230, Loss 0.004021, PSNR 23.956173
Step 240, Loss 0.003842, PSNR 24.154022
Step 250, L

In [397]:
losses["relu_v"], psnrs["relu_v"], outputs["relu_v"], ress["relu_v"], _ = train(relu_v, total_steps, summary_steps, input_type="pv")

Step 0, Loss 0.012740, PSNR 18.948330
Step 10, Loss 0.012296, PSNR 19.102404
Step 20, Loss 0.011858, PSNR 19.259928
Step 30, Loss 0.011419, PSNR 19.423660
Step 40, Loss 0.010987, PSNR 19.591150
Step 50, Loss 0.010561, PSNR 19.763042
Step 60, Loss 0.010139, PSNR 19.939936
Step 70, Loss 0.009720, PSNR 20.123186
Step 80, Loss 0.009302, PSNR 20.314304
Step 90, Loss 0.008887, PSNR 20.512306
Step 100, Loss 0.008482, PSNR 20.715219
Step 110, Loss 0.008086, PSNR 20.922543
Step 120, Loss 0.007701, PSNR 21.134357
Step 130, Loss 0.007329, PSNR 21.349573
Step 140, Loss 0.006968, PSNR 21.569038
Step 150, Loss 0.006618, PSNR 21.792747
Step 160, Loss 0.006280, PSNR 22.020121
Step 170, Loss 0.005957, PSNR 22.249849
Step 180, Loss 0.005646, PSNR 22.482355
Step 190, Loss 0.005349, PSNR 22.716972
Step 200, Loss 0.005070, PSNR 22.949541
Step 210, Loss 0.004814, PSNR 23.174992
Step 220, Loss 0.004581, PSNR 23.390123
Step 230, Loss 0.004373, PSNR 23.592606
Step 240, Loss 0.004188, PSNR 23.779593
Step 250, L

In [398]:
losses["relu_d"], psnrs["relu_d"], outputs["relu_d"], ress["relu_d"], _ = train(relu_d, total_steps, summary_steps, input_type="pd")

Step 0, Loss 0.012099, PSNR 19.172417
Step 10, Loss 0.011701, PSNR 19.317924
Step 20, Loss 0.011308, PSNR 19.466278
Step 30, Loss 0.010921, PSNR 19.617397
Step 40, Loss 0.010537, PSNR 19.772713
Step 50, Loss 0.010158, PSNR 19.931988
Step 60, Loss 0.009785, PSNR 20.094412
Step 70, Loss 0.009418, PSNR 20.260294
Step 80, Loss 0.009056, PSNR 20.430649
Step 90, Loss 0.008698, PSNR 20.606024
Step 100, Loss 0.008344, PSNR 20.786118
Step 110, Loss 0.007995, PSNR 20.971720
Step 120, Loss 0.007651, PSNR 21.162825
Step 130, Loss 0.007313, PSNR 21.358940
Step 140, Loss 0.006983, PSNR 21.559849
Step 150, Loss 0.006661, PSNR 21.764812
Step 160, Loss 0.006348, PSNR 21.973320
Step 170, Loss 0.006047, PSNR 22.184383
Step 180, Loss 0.005758, PSNR 22.396927
Step 190, Loss 0.005483, PSNR 22.609732
Step 200, Loss 0.005223, PSNR 22.821150
Step 210, Loss 0.004978, PSNR 23.029022
Step 220, Loss 0.004753, PSNR 23.230675
Step 230, Loss 0.004547, PSNR 23.422842
Step 240, Loss 0.004359, PSNR 23.605824
Step 250, L

In [399]:
losses["relu_e"], psnrs["relu_e"], outputs["relu_e"], ress["relu_e"], _ = train(relu_e, total_steps, summary_steps, input_type="pe")

Step 0, Loss 0.010314, PSNR 19.865929
Step 10, Loss 0.009951, PSNR 20.021200
Step 20, Loss 0.009598, PSNR 20.178415
Step 30, Loss 0.009252, PSNR 20.337790
Step 40, Loss 0.008912, PSNR 20.500479
Step 50, Loss 0.008575, PSNR 20.667469
Step 60, Loss 0.008238, PSNR 20.841965
Step 70, Loss 0.007905, PSNR 21.021130
Step 80, Loss 0.007577, PSNR 21.204990
Step 90, Loss 0.007254, PSNR 21.394405
Step 100, Loss 0.006936, PSNR 21.589100
Step 110, Loss 0.006628, PSNR 21.786118
Step 120, Loss 0.006331, PSNR 21.984955
Step 130, Loss 0.006047, PSNR 22.184959
Step 140, Loss 0.005773, PSNR 22.386253
Step 150, Loss 0.005512, PSNR 22.586725
Step 160, Loss 0.005267, PSNR 22.784739
Step 170, Loss 0.005036, PSNR 22.979097
Step 180, Loss 0.004823, PSNR 23.167244
Step 190, Loss 0.004627, PSNR 23.347281
Step 200, Loss 0.004448, PSNR 23.518829
Step 210, Loss 0.004284, PSNR 23.681061
Step 220, Loss 0.004139, PSNR 23.831108
Step 230, Loss 0.004013, PSNR 23.964836
Step 240, Loss 0.003907, PSNR 24.081059
Step 250, L

In [400]:
losses["relu_pv"], psnrs["relu_pv"], outputs["relu_pv"], ress["relu_pv"], _ = train(relu_pv, total_steps, summary_steps, input_type="pv")

Step 0, Loss 0.011481, PSNR 19.400173
Step 10, Loss 0.010974, PSNR 19.596155
Step 20, Loss 0.010472, PSNR 19.799694
Step 30, Loss 0.009966, PSNR 20.014736
Step 40, Loss 0.009455, PSNR 20.243389
Step 50, Loss 0.008943, PSNR 20.485374
Step 60, Loss 0.008427, PSNR 20.743074
Step 70, Loss 0.007911, PSNR 21.017828
Step 80, Loss 0.007398, PSNR 21.308720
Step 90, Loss 0.006893, PSNR 21.615610
Step 100, Loss 0.006405, PSNR 21.935051
Step 110, Loss 0.005938, PSNR 22.263760
Step 120, Loss 0.005496, PSNR 22.599728
Step 130, Loss 0.005085, PSNR 22.936703
Step 140, Loss 0.004718, PSNR 23.262238
Step 150, Loss 0.004398, PSNR 23.567276
Step 160, Loss 0.004126, PSNR 23.844509
Step 170, Loss 0.003901, PSNR 24.088245
Step 180, Loss 0.003720, PSNR 24.294300
Step 190, Loss 0.003579, PSNR 24.462347
Step 200, Loss 0.003472, PSNR 24.594641
Step 210, Loss 0.003392, PSNR 24.694921
Step 220, Loss 0.003335, PSNR 24.768648
Step 230, Loss 0.003295, PSNR 24.821722
Step 240, Loss 0.003266, PSNR 24.860247
Step 250, L

In [401]:
losses["relu_pd"], psnrs["relu_pd"], outputs["relu_pd"], ress["relu_pd"], _ = train(relu_pd, total_steps, summary_steps, input_type="pd")

Step 0, Loss 0.012176, PSNR 19.145012
Step 10, Loss 0.011549, PSNR 19.374521
Step 20, Loss 0.010932, PSNR 19.612829
Step 30, Loss 0.010322, PSNR 19.862345
Step 40, Loss 0.009723, PSNR 20.121983
Step 50, Loss 0.009139, PSNR 20.391047
Step 60, Loss 0.008565, PSNR 20.672552
Step 70, Loss 0.008003, PSNR 20.967251
Step 80, Loss 0.007457, PSNR 21.274101
Step 90, Loss 0.006933, PSNR 21.590565
Step 100, Loss 0.006436, PSNR 21.913851
Step 110, Loss 0.005972, PSNR 22.238894
Step 120, Loss 0.005544, PSNR 22.562124
Step 130, Loss 0.005153, PSNR 22.879189
Step 140, Loss 0.004803, PSNR 23.184788
Step 150, Loss 0.004494, PSNR 23.473316
Step 160, Loss 0.004227, PSNR 23.739311
Step 170, Loss 0.004001, PSNR 23.978157
Step 180, Loss 0.003815, PSNR 24.185158
Step 190, Loss 0.003666, PSNR 24.357983
Step 200, Loss 0.003550, PSNR 24.497498
Step 210, Loss 0.003462, PSNR 24.606388
Step 220, Loss 0.003397, PSNR 24.688923
Step 230, Loss 0.003350, PSNR 24.749649
Step 240, Loss 0.003316, PSNR 24.793400
Step 250, L

In [402]:
losses["relu_pe"], psnrs["relu_pe"], outputs["relu_pe"], ress["relu_pe"], _ = train(relu_pe, total_steps, summary_steps, input_type="pe")

Step 0, Loss 0.013875, PSNR 18.577650
Step 10, Loss 0.013251, PSNR 18.777412
Step 20, Loss 0.012645, PSNR 18.980778
Step 30, Loss 0.012053, PSNR 19.188879
Step 40, Loss 0.011476, PSNR 19.402197
Step 50, Loss 0.010908, PSNR 19.622601
Step 60, Loss 0.010349, PSNR 19.851116
Step 70, Loss 0.009801, PSNR 20.087490
Step 80, Loss 0.009266, PSNR 20.330908
Step 90, Loss 0.008738, PSNR 20.585796
Step 100, Loss 0.008217, PSNR 20.853004
Step 110, Loss 0.007709, PSNR 21.129772
Step 120, Loss 0.007218, PSNR 21.415703
Step 130, Loss 0.006744, PSNR 21.710768
Step 140, Loss 0.006290, PSNR 22.013172
Step 150, Loss 0.005861, PSNR 22.320122
Step 160, Loss 0.005460, PSNR 22.627892
Step 170, Loss 0.005091, PSNR 22.932056
Step 180, Loss 0.004758, PSNR 23.226124
Step 190, Loss 0.004462, PSNR 23.504501
Step 200, Loss 0.004204, PSNR 23.763781
Step 210, Loss 0.003984, PSNR 23.996572
Step 220, Loss 0.003807, PSNR 24.194038
Step 230, Loss 0.003670, PSNR 24.353802
Step 240, Loss 0.003566, PSNR 24.477901
Step 250, L

In [403]:
losses["relu"], psnrs["relu"], outputs["relu"], ress["relu"], _ = train(relu, total_steps, summary_steps, input_type="pvde")

Step 0, Loss 0.011633, PSNR 19.343157
Step 10, Loss 0.010535, PSNR 19.773788
Step 20, Loss 0.009507, PSNR 20.219486
Step 30, Loss 0.008562, PSNR 20.674456
Step 40, Loss 0.007705, PSNR 21.132416
Step 50, Loss 0.006942, PSNR 21.585455
Step 60, Loss 0.006251, PSNR 22.040382
Step 70, Loss 0.005635, PSNR 22.491001
Step 80, Loss 0.005097, PSNR 22.926445
Step 90, Loss 0.004644, PSNR 23.331120
Step 100, Loss 0.004273, PSNR 23.692495
Step 110, Loss 0.003979, PSNR 24.002657
Step 120, Loss 0.003753, PSNR 24.256250
Step 130, Loss 0.003590, PSNR 24.449650
Step 140, Loss 0.003479, PSNR 24.585808
Step 150, Loss 0.003407, PSNR 24.676817
Step 160, Loss 0.003360, PSNR 24.736595
Step 170, Loss 0.003329, PSNR 24.776218
Step 180, Loss 0.003309, PSNR 24.803505
Step 190, Loss 0.003293, PSNR 24.824102
Step 200, Loss 0.003280, PSNR 24.841215
Step 210, Loss 0.003268, PSNR 24.856779
Step 220, Loss 0.003257, PSNR 24.871637
Step 230, Loss 0.003246, PSNR 24.886263
Step 240, Loss 0.003236, PSNR 24.900383
Step 250, L

### FFN

In [404]:
losses["ff_p"], psnrs["ff_p"], outputs["ff_p"], ress["ff_p"], _ = train(ff_p, total_steps, summary_steps, input_type="pv")

Step 0, Loss 0.013468, PSNR 18.706957
Step 10, Loss 0.013252, PSNR 18.777279
Step 20, Loss 0.013037, PSNR 18.848150
Step 30, Loss 0.012824, PSNR 18.919842
Step 40, Loss 0.012610, PSNR 18.992765
Step 50, Loss 0.012395, PSNR 19.067417
Step 60, Loss 0.012178, PSNR 19.144335
Step 70, Loss 0.011956, PSNR 19.224056
Step 80, Loss 0.011730, PSNR 19.307138
Step 90, Loss 0.011497, PSNR 19.394125
Step 100, Loss 0.011257, PSNR 19.485603
Step 110, Loss 0.011010, PSNR 19.582184
Step 120, Loss 0.010754, PSNR 19.684464
Step 130, Loss 0.010488, PSNR 19.793051
Step 140, Loss 0.010213, PSNR 19.908600
Step 150, Loss 0.009927, PSNR 20.031759
Step 160, Loss 0.009631, PSNR 20.163176
Step 170, Loss 0.009325, PSNR 20.303488
Step 180, Loss 0.009009, PSNR 20.453264
Step 190, Loss 0.008684, PSNR 20.613018
Step 200, Loss 0.008350, PSNR 20.783180
Step 210, Loss 0.008009, PSNR 20.964048
Step 220, Loss 0.007663, PSNR 21.155798
Step 230, Loss 0.007314, PSNR 21.358311
Step 240, Loss 0.006964, PSNR 21.571215
Step 250, L

In [405]:
losses["ff_v"], psnrs["ff_v"], outputs["ff_v"], ress["ff_v"], _ = train(ff_v, total_steps, summary_steps, input_type="pv")

Step 0, Loss 0.012038, PSNR 19.194569
Step 10, Loss 0.011702, PSNR 19.317430
Step 20, Loss 0.011378, PSNR 19.439489
Step 30, Loss 0.011063, PSNR 19.561083
Step 40, Loss 0.010759, PSNR 19.682434
Step 50, Loss 0.010456, PSNR 19.806255
Step 60, Loss 0.010153, PSNR 19.934185
Step 70, Loss 0.009847, PSNR 20.067139
Step 80, Loss 0.009535, PSNR 20.206968
Step 90, Loss 0.009215, PSNR 20.355223
Step 100, Loss 0.008887, PSNR 20.512255
Step 110, Loss 0.008555, PSNR 20.677881
Step 120, Loss 0.008216, PSNR 20.853477
Step 130, Loss 0.007870, PSNR 21.039997
Step 140, Loss 0.007522, PSNR 21.236954
Step 150, Loss 0.007169, PSNR 21.445648
Step 160, Loss 0.006815, PSNR 21.665512
Step 170, Loss 0.006465, PSNR 21.894281
Step 180, Loss 0.006125, PSNR 22.128716
Step 190, Loss 0.005801, PSNR 22.365234
Step 200, Loss 0.005497, PSNR 22.598907
Step 210, Loss 0.005219, PSNR 22.824402
Step 220, Loss 0.004971, PSNR 23.035851
Step 230, Loss 0.004754, PSNR 23.229042
Step 240, Loss 0.004569, PSNR 23.402109
Step 250, L

In [406]:
losses["ff_d"], psnrs["ff_d"], outputs["ff_d"], ress["ff_d"], _ = train(ff_d, total_steps, summary_steps, input_type="pd")

Step 0, Loss 0.012820, PSNR 18.921288
Step 10, Loss 0.012456, PSNR 19.046286
Step 20, Loss 0.012091, PSNR 19.175251
Step 30, Loss 0.011725, PSNR 19.309017
Step 40, Loss 0.011353, PSNR 19.448887
Step 50, Loss 0.010972, PSNR 19.597330
Step 60, Loss 0.010577, PSNR 19.756433
Step 70, Loss 0.010168, PSNR 19.927736
Step 80, Loss 0.009742, PSNR 20.113401
Step 90, Loss 0.009300, PSNR 20.315401
Step 100, Loss 0.008842, PSNR 20.534367
Step 110, Loss 0.008374, PSNR 20.770884
Step 120, Loss 0.007898, PSNR 21.024561
Step 130, Loss 0.007421, PSNR 21.295246
Step 140, Loss 0.006949, PSNR 21.580942
Step 150, Loss 0.006490, PSNR 21.877647
Step 160, Loss 0.006054, PSNR 22.179436
Step 170, Loss 0.005649, PSNR 22.480148
Step 180, Loss 0.005281, PSNR 22.773216
Step 190, Loss 0.004952, PSNR 23.051929
Step 200, Loss 0.004666, PSNR 23.310848
Step 210, Loss 0.004420, PSNR 23.545635
Step 220, Loss 0.004213, PSNR 23.753622
Step 230, Loss 0.004042, PSNR 23.933580
Step 240, Loss 0.003904, PSNR 24.085302
Step 250, L

In [407]:
losses["ff_e"], psnrs["ff_e"], outputs["ff_e"], ress["ff_e"], _ = train(ff_e, total_steps, summary_steps, input_type="pe")

Step 0, Loss 0.013654, PSNR 18.647390
Step 10, Loss 0.013385, PSNR 18.733847
Step 20, Loss 0.013119, PSNR 18.820869
Step 30, Loss 0.012856, PSNR 18.908813
Step 40, Loss 0.012594, PSNR 18.998533
Step 50, Loss 0.012327, PSNR 19.091259
Step 60, Loss 0.012056, PSNR 19.187849
Step 70, Loss 0.011774, PSNR 19.290876
Step 80, Loss 0.011479, PSNR 19.400990
Step 90, Loss 0.011169, PSNR 19.519817
Step 100, Loss 0.010845, PSNR 19.647892
Step 110, Loss 0.010505, PSNR 19.786114
Step 120, Loss 0.010150, PSNR 19.935278
Step 130, Loss 0.009781, PSNR 20.095970
Step 140, Loss 0.009398, PSNR 20.269711
Step 150, Loss 0.009003, PSNR 20.456182
Step 160, Loss 0.008599, PSNR 20.655296
Step 170, Loss 0.008190, PSNR 20.866970
Step 180, Loss 0.007779, PSNR 21.090561
Step 190, Loss 0.007371, PSNR 21.325006
Step 200, Loss 0.006968, PSNR 21.568779
Step 210, Loss 0.006577, PSNR 21.819637
Step 220, Loss 0.006202, PSNR 22.074593
Step 230, Loss 0.005848, PSNR 22.330032
Step 240, Loss 0.005517, PSNR 22.582579
Step 250, L

In [408]:
losses["ff_pv"], psnrs["ff_pv"], outputs["ff_pv"], ress["ff_pv"], _ = train(ff_pv, total_steps, summary_steps, input_type="pv")

Step 0, Loss 0.012961, PSNR 18.873573
Step 10, Loss 0.012549, PSNR 19.014006
Step 20, Loss 0.012146, PSNR 19.155561
Step 30, Loss 0.011751, PSNR 19.299234
Step 40, Loss 0.011359, PSNR 19.446514
Step 50, Loss 0.010966, PSNR 19.599350
Step 60, Loss 0.010569, PSNR 19.759754
Step 70, Loss 0.010163, PSNR 19.929651
Step 80, Loss 0.009748, PSNR 20.110788
Step 90, Loss 0.009322, PSNR 20.305126
Step 100, Loss 0.008883, PSNR 20.514168
Step 110, Loss 0.008435, PSNR 20.739029
Step 120, Loss 0.007979, PSNR 20.980412
Step 130, Loss 0.007519, PSNR 21.238342
Step 140, Loss 0.007061, PSNR 21.511566
Step 150, Loss 0.006610, PSNR 21.797768
Step 160, Loss 0.006176, PSNR 22.092884
Step 170, Loss 0.005766, PSNR 22.390995
Step 180, Loss 0.005390, PSNR 22.684259
Step 190, Loss 0.005054, PSNR 22.963541
Step 200, Loss 0.004765, PSNR 23.219635
Step 210, Loss 0.004524, PSNR 23.445223
Step 220, Loss 0.004329, PSNR 23.636459
Step 230, Loss 0.004175, PSNR 23.793854
Step 240, Loss 0.004054, PSNR 23.921335
Step 250, L

In [409]:
losses["ff_pd"], psnrs["ff_pd"], outputs["ff_pd"], ress["ff_pd"], _ = train(ff_pd, total_steps, summary_steps, input_type="pd")

Step 0, Loss 0.012075, PSNR 19.181025
Step 10, Loss 0.011652, PSNR 19.335995
Step 20, Loss 0.011228, PSNR 19.497044
Step 30, Loss 0.010799, PSNR 19.666000
Step 40, Loss 0.010362, PSNR 19.845449
Step 50, Loss 0.009914, PSNR 20.037710
Step 60, Loss 0.009451, PSNR 20.245008
Step 70, Loss 0.008975, PSNR 20.469645
Step 80, Loss 0.008484, PSNR 20.713911
Step 90, Loss 0.007981, PSNR 20.979496
Step 100, Loss 0.007469, PSNR 21.267345
Step 110, Loss 0.006955, PSNR 21.577255
Step 120, Loss 0.006446, PSNR 21.907257
Step 130, Loss 0.005953, PSNR 22.252815
Step 140, Loss 0.005487, PSNR 22.606756
Step 150, Loss 0.005060, PSNR 22.958355
Step 160, Loss 0.004684, PSNR 23.293798
Step 170, Loss 0.004367, PSNR 23.598137
Step 180, Loss 0.004112, PSNR 23.859146
Step 190, Loss 0.003917, PSNR 24.070278
Step 200, Loss 0.003773, PSNR 24.232574
Step 210, Loss 0.003670, PSNR 24.352833
Step 220, Loss 0.003597, PSNR 24.440496
Step 230, Loss 0.003544, PSNR 24.504700
Step 240, Loss 0.003505, PSNR 24.552666
Step 250, L

In [410]:
losses["ff_pe"], psnrs["ff_pe"], outputs["ff_pe"], ress["ff_pe"], _ = train(ff_pe, total_steps, summary_steps, input_type="pe")

Step 0, Loss 0.013028, PSNR 18.851215
Step 10, Loss 0.012609, PSNR 18.993164
Step 20, Loss 0.012197, PSNR 19.137520
Step 30, Loss 0.011789, PSNR 19.285229
Step 40, Loss 0.011381, PSNR 19.438040
Step 50, Loss 0.010970, PSNR 19.597878
Step 60, Loss 0.010551, PSNR 19.766953
Step 70, Loss 0.010122, PSNR 19.947540
Step 80, Loss 0.009679, PSNR 20.141876
Step 90, Loss 0.009222, PSNR 20.351875
Step 100, Loss 0.008751, PSNR 20.579462
Step 110, Loss 0.008268, PSNR 20.826256
Step 120, Loss 0.007775, PSNR 21.093081
Step 130, Loss 0.007278, PSNR 21.379784
Step 140, Loss 0.006784, PSNR 21.685068
Step 150, Loss 0.006301, PSNR 22.005856
Step 160, Loss 0.005840, PSNR 22.336235
Step 170, Loss 0.005410, PSNR 22.667694
Step 180, Loss 0.005023, PSNR 22.989956
Step 190, Loss 0.004686, PSNR 23.291771
Step 200, Loss 0.004403, PSNR 23.562691
Step 210, Loss 0.004173, PSNR 23.795837
Step 220, Loss 0.003991, PSNR 23.988993
Step 230, Loss 0.003851, PSNR 24.144306
Step 240, Loss 0.003744, PSNR 24.266754
Step 250, L

In [411]:
losses["ff"], psnrs["ff"], outputs["ff"], ress["ff"], _ = train(ff, total_steps, summary_steps, input_type="pvde")

Step 0, Loss 0.013578, PSNR 18.671696
Step 10, Loss 0.012748, PSNR 18.945442
Step 20, Loss 0.011935, PSNR 19.231636
Step 30, Loss 0.011127, PSNR 19.536165
Step 40, Loss 0.010316, PSNR 19.864822
Step 50, Loss 0.009493, PSNR 20.225746
Step 60, Loss 0.008658, PSNR 20.626070
Step 70, Loss 0.007814, PSNR 21.071095
Step 80, Loss 0.006981, PSNR 21.560934
Step 90, Loss 0.006187, PSNR 22.084959
Step 100, Loss 0.005469, PSNR 22.621246
Step 110, Loss 0.004859, PSNR 23.134510
Step 120, Loss 0.004385, PSNR 23.579914
Step 130, Loss 0.004052, PSNR 23.923729
Step 140, Loss 0.003839, PSNR 24.157265
Step 150, Loss 0.003716, PSNR 24.299505
Step 160, Loss 0.003645, PSNR 24.382793
Step 170, Loss 0.003602, PSNR 24.434855
Step 180, Loss 0.003571, PSNR 24.472298
Step 190, Loss 0.003546, PSNR 24.503061
Step 200, Loss 0.003523, PSNR 24.530704
Step 210, Loss 0.003502, PSNR 24.556982
Step 220, Loss 0.003481, PSNR 24.582489
Step 230, Loss 0.003461, PSNR 24.607368
Step 240, Loss 0.003442, PSNR 24.631617
Step 250, L

### Gauss

In [412]:
losses["gauss_p"], psnrs["gauss_p"], outputs["gauss_p"], ress["gauss_p"], _ = train(gauss_p, total_steps, summary_steps, input_type="pv")

Step 0, Loss 0.012376, PSNR 19.074312
Step 10, Loss 0.010586, PSNR 19.752596
Step 20, Loss 0.008952, PSNR 20.481035
Step 30, Loss 0.007525, PSNR 21.234707
Step 40, Loss 0.006333, PSNR 21.983879
Step 50, Loss 0.005375, PSNR 22.696480
Step 60, Loss 0.004635, PSNR 23.339600
Step 70, Loss 0.004086, PSNR 23.886806
Step 80, Loss 0.003693, PSNR 24.326012
Step 90, Loss 0.003419, PSNR 24.660885
Step 100, Loss 0.003231, PSNR 24.906790
Step 110, Loss 0.003102, PSNR 25.084213
Step 120, Loss 0.003011, PSNR 25.212687
Step 130, Loss 0.002946, PSNR 25.307449
Step 140, Loss 0.002898, PSNR 25.379124
Step 150, Loss 0.002861, PSNR 25.434837
Step 160, Loss 0.002832, PSNR 25.479370
Step 170, Loss 0.002808, PSNR 25.515934
Step 180, Loss 0.002788, PSNR 25.546700
Step 190, Loss 0.002771, PSNR 25.573149
Step 200, Loss 0.002757, PSNR 25.596302
Step 210, Loss 0.002744, PSNR 25.616877
Step 220, Loss 0.002732, PSNR 25.635387
Step 230, Loss 0.002721, PSNR 25.652206
Step 240, Loss 0.002712, PSNR 25.667606
Step 250, L

In [413]:
losses["gauss_v"], psnrs["gauss_v"], outputs["gauss_v"], ress["gauss_v"], _ = train(gauss_v, total_steps, summary_steps, input_type="pv")

Step 0, Loss 0.015367, PSNR 18.134001
Step 10, Loss 0.011782, PSNR 19.287680
Step 20, Loss 0.008380, PSNR 20.767548
Step 30, Loss 0.005818, PSNR 22.352108
Step 40, Loss 0.004463, PSNR 23.503933
Step 50, Loss 0.003823, PSNR 24.176199
Step 60, Loss 0.003475, PSNR 24.590977
Step 70, Loss 0.003277, PSNR 24.845285
Step 80, Loss 0.003149, PSNR 25.018541
Step 90, Loss 0.003062, PSNR 25.140314
Step 100, Loss 0.002999, PSNR 25.229548
Step 110, Loss 0.002952, PSNR 25.298157
Step 120, Loss 0.002917, PSNR 25.351315
Step 130, Loss 0.002888, PSNR 25.393476
Step 140, Loss 0.002866, PSNR 25.427711
Step 150, Loss 0.002847, PSNR 25.456108
Step 160, Loss 0.002831, PSNR 25.480007
Step 170, Loss 0.002818, PSNR 25.500271
Step 180, Loss 0.002807, PSNR 25.517540
Step 190, Loss 0.002797, PSNR 25.532370
Step 200, Loss 0.002789, PSNR 25.545261
Step 210, Loss 0.002782, PSNR 25.556650
Step 220, Loss 0.002775, PSNR 25.566883
Step 230, Loss 0.002769, PSNR 25.576199
Step 240, Loss 0.002764, PSNR 25.584686
Step 250, L

In [414]:
losses["gauss_d"], psnrs["gauss_d"], outputs["gauss_d"], ress["gauss_d"], _ = train(gauss_d, total_steps, summary_steps, input_type="pd")

Step 0, Loss 0.012523, PSNR 19.022877
Step 10, Loss 0.009658, PSNR 20.151148
Step 20, Loss 0.007329, PSNR 21.349674
Step 30, Loss 0.005692, PSNR 22.447124
Step 40, Loss 0.004693, PSNR 23.285435
Step 50, Loss 0.004121, PSNR 23.850157
Step 60, Loss 0.003801, PSNR 24.200474
Step 70, Loss 0.003625, PSNR 24.407185
Step 80, Loss 0.003521, PSNR 24.533199
Step 90, Loss 0.003454, PSNR 24.616358
Step 100, Loss 0.003409, PSNR 24.673788
Step 110, Loss 0.003378, PSNR 24.713917
Step 120, Loss 0.003356, PSNR 24.741934
Step 130, Loss 0.003341, PSNR 24.761370
Step 140, Loss 0.003331, PSNR 24.774878
Step 150, Loss 0.003323, PSNR 24.784431
Step 160, Loss 0.003318, PSNR 24.791410
Step 170, Loss 0.003314, PSNR 24.796713
Step 180, Loss 0.003311, PSNR 24.800890
Step 190, Loss 0.003308, PSNR 24.804264
Step 200, Loss 0.003306, PSNR 24.807035
Step 210, Loss 0.003304, PSNR 24.809334
Step 220, Loss 0.003303, PSNR 24.811256
Step 230, Loss 0.003302, PSNR 24.812878
Step 240, Loss 0.003300, PSNR 24.814255
Step 250, L

In [415]:
losses["gauss_e"], psnrs["gauss_e"], outputs["gauss_e"], ress["gauss_e"], _ = train(gauss_e, total_steps, summary_steps, input_type="pe")

Step 0, Loss 0.013975, PSNR 18.546545
Step 10, Loss 0.010875, PSNR 19.635563
Step 20, Loss 0.008231, PSNR 20.845596
Step 30, Loss 0.006189, PSNR 22.083668
Step 40, Loss 0.004813, PSNR 23.175489
Step 50, Loss 0.004041, PSNR 23.935270
Step 60, Loss 0.003682, PSNR 24.339067
Step 70, Loss 0.003531, PSNR 24.520721
Step 80, Loss 0.003462, PSNR 24.607079
Step 90, Loss 0.003423, PSNR 24.656096
Step 100, Loss 0.003398, PSNR 24.688316
Step 110, Loss 0.003379, PSNR 24.711477
Step 120, Loss 0.003366, PSNR 24.728767
Step 130, Loss 0.003356, PSNR 24.741894
Step 140, Loss 0.003348, PSNR 24.752151
Step 150, Loss 0.003342, PSNR 24.760410
Step 160, Loss 0.003336, PSNR 24.767218
Step 170, Loss 0.003332, PSNR 24.772947
Step 180, Loss 0.003328, PSNR 24.777842
Step 190, Loss 0.003325, PSNR 24.782057
Step 200, Loss 0.003322, PSNR 24.785706
Step 210, Loss 0.003320, PSNR 24.788874
Step 220, Loss 0.003318, PSNR 24.791626
Step 230, Loss 0.003316, PSNR 24.794024
Step 240, Loss 0.003314, PSNR 24.796106
Step 250, L

In [416]:
losses["gauss_pv"], psnrs["gauss_pv"], outputs["gauss_pv"], ress["gauss_pv"], _ = train(gauss_pv, total_steps, summary_steps, input_type="pv")

Step 0, Loss 0.011537, PSNR 19.379135
Step 10, Loss 0.008333, PSNR 20.791914
Step 20, Loss 0.005912, PSNR 22.282795
Step 30, Loss 0.004483, PSNR 23.484594
Step 40, Loss 0.003759, PSNR 24.249828
Step 50, Loss 0.003294, PSNR 24.822733
Step 60, Loss 0.002933, PSNR 25.326878
Step 70, Loss 0.002658, PSNR 25.754436
Step 80, Loss 0.002439, PSNR 26.127323
Step 90, Loss 0.002263, PSNR 26.453379
Step 100, Loss 0.002118, PSNR 26.739996
Step 110, Loss 0.001998, PSNR 26.995115
Step 120, Loss 0.001895, PSNR 27.223347
Step 130, Loss 0.001808, PSNR 27.428091
Step 140, Loss 0.001733, PSNR 27.612556
Step 150, Loss 0.001667, PSNR 27.780075
Step 160, Loss 0.001609, PSNR 27.933456
Step 170, Loss 0.001558, PSNR 28.074810
Step 180, Loss 0.001512, PSNR 28.205639
Step 190, Loss 0.001470, PSNR 28.327105
Step 200, Loss 0.001432, PSNR 28.440210
Step 210, Loss 0.001398, PSNR 28.545872
Step 220, Loss 0.001366, PSNR 28.644917
Step 230, Loss 0.001337, PSNR 28.738094
Step 240, Loss 0.001310, PSNR 28.826038
Step 250, L

In [417]:
losses["gauss_pd"], psnrs["gauss_pd"], outputs["gauss_pd"], ress["gauss_pd"], _ = train(gauss_pd, total_steps, summary_steps, input_type="pd")

Step 0, Loss 0.010504, PSNR 19.786560
Step 10, Loss 0.007545, PSNR 21.223583
Step 20, Loss 0.005509, PSNR 22.589293
Step 30, Loss 0.004346, PSNR 23.618925
Step 40, Loss 0.003745, PSNR 24.265234
Step 50, Loss 0.003391, PSNR 24.696556
Step 60, Loss 0.003176, PSNR 24.981125
Step 70, Loss 0.003051, PSNR 25.155716
Step 80, Loss 0.002971, PSNR 25.271168
Step 90, Loss 0.002916, PSNR 25.351782
Step 100, Loss 0.002876, PSNR 25.411648
Step 110, Loss 0.002845, PSNR 25.458702
Step 120, Loss 0.002820, PSNR 25.497141
Step 130, Loss 0.002799, PSNR 25.529362
Step 140, Loss 0.002782, PSNR 25.556919
Step 150, Loss 0.002766, PSNR 25.580952
Step 160, Loss 0.002753, PSNR 25.602253
Step 170, Loss 0.002741, PSNR 25.621378
Step 180, Loss 0.002730, PSNR 25.638733
Step 190, Loss 0.002720, PSNR 25.654612
Step 200, Loss 0.002711, PSNR 25.669233
Step 210, Loss 0.002702, PSNR 25.682772
Step 220, Loss 0.002694, PSNR 25.695354
Step 230, Loss 0.002687, PSNR 25.707096
Step 240, Loss 0.002680, PSNR 25.718086
Step 250, L

In [418]:
losses["gauss_pe"], psnrs["gauss_pe"], outputs["gauss_pe"], ress["gauss_pe"], _ = train(gauss_pe, total_steps, summary_steps, input_type="pe")

Step 0, Loss 0.015231, PSNR 18.172611
Step 10, Loss 0.011858, PSNR 19.260014
Step 20, Loss 0.008969, PSNR 20.472675
Step 30, Loss 0.006733, PSNR 21.718218
Step 40, Loss 0.005176, PSNR 22.860186
Step 50, Loss 0.004207, PSNR 23.760632
Step 60, Loss 0.003666, PSNR 24.357899
Step 70, Loss 0.003377, PSNR 24.714943
Step 80, Loss 0.003210, PSNR 24.934883
Step 90, Loss 0.003105, PSNR 25.079538
Step 100, Loss 0.003033, PSNR 25.181881
Step 110, Loss 0.002979, PSNR 25.259689
Step 120, Loss 0.002937, PSNR 25.321157
Step 130, Loss 0.002903, PSNR 25.371098
Step 140, Loss 0.002876, PSNR 25.412739
Step 150, Loss 0.002852, PSNR 25.448187
Step 160, Loss 0.002832, PSNR 25.478859
Step 170, Loss 0.002815, PSNR 25.505697
Step 180, Loss 0.002799, PSNR 25.529419
Step 190, Loss 0.002786, PSNR 25.550610
Step 200, Loss 0.002773, PSNR 25.569740
Step 210, Loss 0.002762, PSNR 25.587168
Step 220, Loss 0.002752, PSNR 25.603163
Step 230, Loss 0.002743, PSNR 25.617928
Step 240, Loss 0.002734, PSNR 25.631634
Step 250, L

In [419]:
losses["gauss"], psnrs["gauss"], outputs["gauss"], ress["gauss"], _ = train(gauss, total_steps, summary_steps, input_type="pvde")

Step 0, Loss 0.016657, PSNR 17.783916
Step 10, Loss 0.010111, PSNR 19.952187
Step 20, Loss 0.005682, PSNR 22.454874
Step 30, Loss 0.003903, PSNR 24.085901
Step 40, Loss 0.003318, PSNR 24.790688
Step 50, Loss 0.002882, PSNR 25.403297
Step 60, Loss 0.002604, PSNR 25.843184
Step 70, Loss 0.002390, PSNR 26.215954
Step 80, Loss 0.002225, PSNR 26.525774
Step 90, Loss 0.002093, PSNR 26.793097
Step 100, Loss 0.001983, PSNR 27.025900
Step 110, Loss 0.001892, PSNR 27.230408
Step 120, Loss 0.001814, PSNR 27.413263
Step 130, Loss 0.001746, PSNR 27.578976
Step 140, Loss 0.001687, PSNR 27.729725
Step 150, Loss 0.001634, PSNR 27.867281
Step 160, Loss 0.001587, PSNR 27.993595
Step 170, Loss 0.001545, PSNR 28.110310
Step 180, Loss 0.001507, PSNR 28.218689
Step 190, Loss 0.001472, PSNR 28.319839
Step 200, Loss 0.001441, PSNR 28.414761
Step 210, Loss 0.001411, PSNR 28.504311
Step 220, Loss 0.001384, PSNR 28.589209
Step 230, Loss 0.001358, PSNR 28.669874
Step 240, Loss 0.001335, PSNR 28.746384
Step 250, L

In [420]:
losses["ff_gauss_p"], psnrs["ff_gauss_p"], outputs["ff_gauss_p"], ress["ff_gauss_p"], _ = train(ff_gauss_p, total_steps, summary_steps, input_type="pv")

Step 0, Loss 0.015080, PSNR 18.215916
Step 10, Loss 0.013447, PSNR 18.713898
Step 20, Loss 0.011915, PSNR 19.239128
Step 30, Loss 0.010520, PSNR 19.779642
Step 40, Loss 0.009271, PSNR 20.328808
Step 50, Loss 0.008163, PSNR 20.881603
Step 60, Loss 0.007189, PSNR 21.433535
Step 70, Loss 0.006338, PSNR 21.980556
Step 80, Loss 0.005600, PSNR 22.518053
Step 90, Loss 0.004966, PSNR 23.040342
Step 100, Loss 0.004424, PSNR 23.541416
Step 110, Loss 0.003966, PSNR 24.016071
Step 120, Loss 0.003580, PSNR 24.461220
Step 130, Loss 0.003254, PSNR 24.875908
Step 140, Loss 0.002978, PSNR 25.260456
Step 150, Loss 0.002744, PSNR 25.615721
Step 160, Loss 0.002545, PSNR 25.942991
Step 170, Loss 0.002375, PSNR 26.244213
Step 180, Loss 0.002228, PSNR 26.521767
Step 190, Loss 0.002100, PSNR 26.778028
Step 200, Loss 0.001988, PSNR 27.015110
Step 210, Loss 0.001890, PSNR 27.234915
Step 220, Loss 0.001803, PSNR 27.439289
Step 230, Loss 0.001726, PSNR 27.629982
Step 240, Loss 0.001656, PSNR 27.808533
Step 250, L

In [421]:
losses["ff_gauss_v"], psnrs["ff_gauss_v"], outputs["ff_gauss_v"], ress["ff_gauss_v"], _ = train(ff_gauss_v, total_steps, summary_steps, input_type="pv")

Step 0, Loss 0.010169, PSNR 19.927113
Step 10, Loss 0.004891, PSNR 23.106014
Step 20, Loss 0.003750, PSNR 24.259508
Step 30, Loss 0.003324, PSNR 24.783270
Step 40, Loss 0.003103, PSNR 25.082298
Step 50, Loss 0.002982, PSNR 25.254604
Step 60, Loss 0.002911, PSNR 25.360090
Step 70, Loss 0.002863, PSNR 25.432100
Step 80, Loss 0.002830, PSNR 25.482716
Step 90, Loss 0.002806, PSNR 25.519773
Step 100, Loss 0.002788, PSNR 25.547840
Step 110, Loss 0.002774, PSNR 25.569294
Step 120, Loss 0.002763, PSNR 25.585884
Step 130, Loss 0.002755, PSNR 25.599010
Step 140, Loss 0.002748, PSNR 25.609642
Step 150, Loss 0.002743, PSNR 25.618420
Step 160, Loss 0.002738, PSNR 25.625748
Step 170, Loss 0.002734, PSNR 25.631920
Step 180, Loss 0.002731, PSNR 25.637177
Step 190, Loss 0.002728, PSNR 25.641710
Step 200, Loss 0.002725, PSNR 25.645672
Step 210, Loss 0.002723, PSNR 25.649157
Step 220, Loss 0.002721, PSNR 25.652239
Step 230, Loss 0.002720, PSNR 25.654957
Step 240, Loss 0.002718, PSNR 25.657356
Step 250, L

In [422]:
losses["ff_gauss_d"], psnrs["ff_gauss_d"], outputs["ff_gauss_d"], ress["ff_gauss_d"], _ = train(ff_gauss_d, total_steps, summary_steps, input_type="pd")

Step 0, Loss 0.014019, PSNR 18.532825
Step 10, Loss 0.007337, PSNR 21.344742
Step 20, Loss 0.004857, PSNR 23.136543
Step 30, Loss 0.003870, PSNR 24.123344
Step 40, Loss 0.003510, PSNR 24.547064
Step 50, Loss 0.003378, PSNR 24.713341
Step 60, Loss 0.003323, PSNR 24.785147
Step 70, Loss 0.003296, PSNR 24.820215
Step 80, Loss 0.003281, PSNR 24.839705
Step 90, Loss 0.003272, PSNR 24.852413
Step 100, Loss 0.003265, PSNR 24.861664
Step 110, Loss 0.003260, PSNR 24.868486
Step 120, Loss 0.003256, PSNR 24.873667
Step 130, Loss 0.003253, PSNR 24.877756
Step 140, Loss 0.003250, PSNR 24.881004
Step 150, Loss 0.003248, PSNR 24.883598
Step 160, Loss 0.003247, PSNR 24.885662
Step 170, Loss 0.003245, PSNR 24.887287
Step 180, Loss 0.003244, PSNR 24.888575
Step 190, Loss 0.003244, PSNR 24.889595
Step 200, Loss 0.003243, PSNR 24.890411
Step 210, Loss 0.003243, PSNR 24.891064
Step 220, Loss 0.003242, PSNR 24.891596
Step 230, Loss 0.003242, PSNR 24.892029
Step 240, Loss 0.003242, PSNR 24.892391
Step 250, L

In [423]:
losses["ff_gauss_e"], psnrs["ff_gauss_e"], outputs["ff_gauss_e"], ress["ff_gauss_e"], _ = train(ff_gauss_e, total_steps, summary_steps, input_type="pe")

Step 0, Loss 0.014919, PSNR 18.262539
Step 10, Loss 0.008000, PSNR 20.969275
Step 20, Loss 0.005184, PSNR 22.853001
Step 30, Loss 0.004023, PSNR 23.954979
Step 40, Loss 0.003575, PSNR 24.466742
Step 50, Loss 0.003408, PSNR 24.674717
Step 60, Loss 0.003343, PSNR 24.758305
Step 70, Loss 0.003313, PSNR 24.797836
Step 80, Loss 0.003295, PSNR 24.822063
Step 90, Loss 0.003282, PSNR 24.838503
Step 100, Loss 0.003274, PSNR 24.849842
Step 110, Loss 0.003267, PSNR 24.858070
Step 120, Loss 0.003263, PSNR 24.864452
Step 130, Loss 0.003259, PSNR 24.869717
Step 140, Loss 0.003255, PSNR 24.874077
Step 150, Loss 0.003253, PSNR 24.877626
Step 160, Loss 0.003250, PSNR 24.880507
Step 170, Loss 0.003249, PSNR 24.882847
Step 180, Loss 0.003247, PSNR 24.884741
Step 190, Loss 0.003246, PSNR 24.886276
Step 200, Loss 0.003245, PSNR 24.887527
Step 210, Loss 0.003244, PSNR 24.888557
Step 220, Loss 0.003244, PSNR 24.889416
Step 230, Loss 0.003243, PSNR 24.890144
Step 240, Loss 0.003243, PSNR 24.890759
Step 250, L

In [424]:
losses["ff_gauss_pv"], psnrs["ff_gauss_pv"], outputs["ff_gauss_pv"], ress["ff_gauss_pv"], _ = train(ff_gauss_pv, total_steps, summary_steps, input_type="pv")

Step 0, Loss 0.011900, PSNR 19.244474
Step 10, Loss 0.008465, PSNR 20.723873
Step 20, Loss 0.006233, PSNR 22.053282
Step 30, Loss 0.004823, PSNR 23.167137
Step 40, Loss 0.003981, PSNR 24.000584
Step 50, Loss 0.003460, PSNR 24.609322
Step 60, Loss 0.003093, PSNR 25.096781
Step 70, Loss 0.002808, PSNR 25.515879
Step 80, Loss 0.002578, PSNR 25.886745
Step 90, Loss 0.002387, PSNR 26.221020
Step 100, Loss 0.002225, PSNR 26.525879
Step 110, Loss 0.002086, PSNR 26.806152
Step 120, Loss 0.001965, PSNR 27.065487
Step 130, Loss 0.001859, PSNR 27.306787
Step 140, Loss 0.001765, PSNR 27.532448
Step 150, Loss 0.001681, PSNR 27.744482
Step 160, Loss 0.001605, PSNR 27.944525
Step 170, Loss 0.001537, PSNR 28.133904
Step 180, Loss 0.001474, PSNR 28.313686
Step 190, Loss 0.001418, PSNR 28.484735
Step 200, Loss 0.001365, PSNR 28.647680
Step 210, Loss 0.001317, PSNR 28.802984
Step 220, Loss 0.001273, PSNR 28.951038
Step 230, Loss 0.001232, PSNR 29.092310
Step 240, Loss 0.001195, PSNR 29.227318
Step 250, L

In [425]:
losses["ff_gauss_pd"], psnrs["ff_gauss_pd"], outputs["ff_gauss_pd"], ress["ff_gauss_pd"], _ = train(ff_gauss_pd, total_steps, summary_steps, input_type="pd")

Step 0, Loss 0.007429, PSNR 21.290760
Step 10, Loss 0.005022, PSNR 22.990829
Step 20, Loss 0.003919, PSNR 24.068155
Step 30, Loss 0.003367, PSNR 24.727394
Step 40, Loss 0.002998, PSNR 25.231846
Step 50, Loss 0.002712, PSNR 25.666954
Step 60, Loss 0.002483, PSNR 26.049767
Step 70, Loss 0.002294, PSNR 26.393856
Step 80, Loss 0.002135, PSNR 26.706043
Step 90, Loss 0.002000, PSNR 26.990768
Step 100, Loss 0.001883, PSNR 27.252123
Step 110, Loss 0.001781, PSNR 27.493574
Step 120, Loss 0.001691, PSNR 27.718008
Step 130, Loss 0.001612, PSNR 27.927677
Step 140, Loss 0.001540, PSNR 28.124266
Step 150, Loss 0.001476, PSNR 28.309067
Step 160, Loss 0.001418, PSNR 28.483095
Step 170, Loss 0.001365, PSNR 28.647194
Step 180, Loss 0.001318, PSNR 28.802107
Step 190, Loss 0.001274, PSNR 28.948536
Step 200, Loss 0.001234, PSNR 29.087162
Step 210, Loss 0.001197, PSNR 29.218624
Step 220, Loss 0.001163, PSNR 29.343510
Step 230, Loss 0.001132, PSNR 29.462345
Step 240, Loss 0.001103, PSNR 29.575594
Step 250, L

In [426]:
losses["ff_gauss_pe"], psnrs["ff_gauss_pe"], outputs["ff_gauss_pe"], ress["ff_gauss_pe"], _ = train(ff_gauss_pe, total_steps, summary_steps, input_type="pe")

Step 0, Loss 0.010568, PSNR 19.760250
Step 10, Loss 0.007451, PSNR 21.278049
Step 20, Loss 0.005483, PSNR 22.609589
Step 30, Loss 0.004304, PSNR 23.661221
Step 40, Loss 0.003618, PSNR 24.415051
Step 50, Loss 0.003184, PSNR 24.970604
Step 60, Loss 0.002872, PSNR 25.417746
Step 70, Loss 0.002629, PSNR 25.801394
Step 80, Loss 0.002432, PSNR 26.139893
Step 90, Loss 0.002267, PSNR 26.444700
Step 100, Loss 0.002127, PSNR 26.722528
Step 110, Loss 0.002006, PSNR 26.977444
Step 120, Loss 0.001900, PSNR 27.212486
Step 130, Loss 0.001807, PSNR 27.430302
Step 140, Loss 0.001725, PSNR 27.633129
Step 150, Loss 0.001651, PSNR 27.822796
Step 160, Loss 0.001585, PSNR 28.000757
Step 170, Loss 0.001525, PSNR 28.168152
Step 180, Loss 0.001470, PSNR 28.325922
Step 190, Loss 0.001421, PSNR 28.474960
Step 200, Loss 0.001375, PSNR 28.616093
Step 210, Loss 0.001334, PSNR 28.750042
Step 220, Loss 0.001295, PSNR 28.877428
Step 230, Loss 0.001259, PSNR 28.998766
Step 240, Loss 0.001226, PSNR 29.114498
Step 250, L

In [427]:
losses["ff_gauss"], psnrs["ff_gauss"], outputs["ff_gauss"], ress["ff_gauss"], _ = train(ff_gauss, total_steps, summary_steps, input_type="pvde")

Step 0, Loss 0.012539, PSNR 19.017530
Step 10, Loss 0.005299, PSNR 22.758341
Step 20, Loss 0.003921, PSNR 24.065674
Step 30, Loss 0.003413, PSNR 24.668150
Step 40, Loss 0.003024, PSNR 25.194286
Step 50, Loss 0.002735, PSNR 25.629976
Step 60, Loss 0.002506, PSNR 26.009550
Step 70, Loss 0.002320, PSNR 26.346033
Step 80, Loss 0.002163, PSNR 26.649694
Step 90, Loss 0.002029, PSNR 26.927032
Step 100, Loss 0.001913, PSNR 27.183048
Step 110, Loss 0.001811, PSNR 27.420906
Step 120, Loss 0.001721, PSNR 27.642897
Step 130, Loss 0.001640, PSNR 27.850952
Step 140, Loss 0.001568, PSNR 28.046705
Step 150, Loss 0.001503, PSNR 28.231535
Step 160, Loss 0.001443, PSNR 28.406563
Step 170, Loss 0.001389, PSNR 28.572758
Step 180, Loss 0.001339, PSNR 28.730953
Step 190, Loss 0.001294, PSNR 28.881876
Step 200, Loss 0.001251, PSNR 29.026155
Step 210, Loss 0.001212, PSNR 29.164326
Step 220, Loss 0.001176, PSNR 29.296848
Step 230, Loss 0.001142, PSNR 29.424118
Step 240, Loss 0.001110, PSNR 29.546492
Step 250, L

### SIREN

In [428]:
losses["siren_p"], psnrs["siren_p"], outputs["siren_p"], ress["siren_p"], _ = train(siren_p, total_steps, summary_steps, input_type="pv")

Step 0, Loss 0.007155, PSNR 21.454199
Step 10, Loss 0.003129, PSNR 25.046356
Step 20, Loss 0.003027, PSNR 25.189812
Step 30, Loss 0.002880, PSNR 25.405861
Step 40, Loss 0.002813, PSNR 25.508287
Step 50, Loss 0.002796, PSNR 25.534481
Step 60, Loss 0.002780, PSNR 25.560066
Step 70, Loss 0.002771, PSNR 25.574360
Step 80, Loss 0.002763, PSNR 25.586452
Step 90, Loss 0.002757, PSNR 25.595837
Step 100, Loss 0.002752, PSNR 25.604034
Step 110, Loss 0.002747, PSNR 25.611313
Step 120, Loss 0.002743, PSNR 25.617865
Step 130, Loss 0.002739, PSNR 25.623840
Step 140, Loss 0.002736, PSNR 25.629351
Step 150, Loss 0.002732, PSNR 25.634459
Step 160, Loss 0.002729, PSNR 25.639219
Step 170, Loss 0.002727, PSNR 25.643675
Step 180, Loss 0.002724, PSNR 25.647861
Step 190, Loss 0.002722, PSNR 25.651808
Step 200, Loss 0.002719, PSNR 25.655546
Step 210, Loss 0.002717, PSNR 25.659100
Step 220, Loss 0.002715, PSNR 25.662489
Step 230, Loss 0.002713, PSNR 25.665739
Step 240, Loss 0.002711, PSNR 25.668871
Step 250, L

In [429]:
losses["siren_v"], psnrs["siren_v"], outputs["siren_v"], ress["siren_v"], _ = train(siren_v, total_steps, summary_steps, input_type="pv")

Step 0, Loss 0.011823, PSNR 19.272739
Step 10, Loss 0.003766, PSNR 24.241751
Step 20, Loss 0.003082, PSNR 25.111876
Step 30, Loss 0.002789, PSNR 25.545073
Step 40, Loss 0.002772, PSNR 25.571651
Step 50, Loss 0.002739, PSNR 25.624699
Step 60, Loss 0.002730, PSNR 25.637796
Step 70, Loss 0.002725, PSNR 25.645741
Step 80, Loss 0.002723, PSNR 25.649685
Step 90, Loss 0.002721, PSNR 25.652412
Step 100, Loss 0.002720, PSNR 25.653955
Step 110, Loss 0.002719, PSNR 25.655132
Step 120, Loss 0.002719, PSNR 25.656094
Step 130, Loss 0.002718, PSNR 25.656902
Step 140, Loss 0.002718, PSNR 25.657593
Step 150, Loss 0.002718, PSNR 25.658186
Step 160, Loss 0.002717, PSNR 25.658705
Step 170, Loss 0.002717, PSNR 25.659164
Step 180, Loss 0.002717, PSNR 25.659571
Step 190, Loss 0.002716, PSNR 25.659931
Step 200, Loss 0.002716, PSNR 25.660248
Step 210, Loss 0.002716, PSNR 25.660532
Step 220, Loss 0.002716, PSNR 25.660784
Step 230, Loss 0.002716, PSNR 25.661007
Step 240, Loss 0.002716, PSNR 25.661205
Step 250, L

In [430]:
losses["siren_d"], psnrs["siren_d"], outputs["siren_d"], ress["siren_d"], _ = train(siren_d, total_steps, summary_steps, input_type="pd")

Step 0, Loss 0.018769, PSNR 17.265657
Step 10, Loss 0.005704, PSNR 22.438143
Step 20, Loss 0.003974, PSNR 24.007692
Step 30, Loss 0.003424, PSNR 24.654577
Step 40, Loss 0.003351, PSNR 24.748230
Step 50, Loss 0.003336, PSNR 24.768381
Step 60, Loss 0.003327, PSNR 24.779806
Step 70, Loss 0.003321, PSNR 24.787390
Step 80, Loss 0.003318, PSNR 24.791889
Step 90, Loss 0.003316, PSNR 24.794237
Step 100, Loss 0.003315, PSNR 24.795723
Step 110, Loss 0.003314, PSNR 24.796923
Step 120, Loss 0.003313, PSNR 24.797947
Step 130, Loss 0.003312, PSNR 24.798880
Step 140, Loss 0.003312, PSNR 24.799740
Step 150, Loss 0.003311, PSNR 24.800552
Step 160, Loss 0.003310, PSNR 24.801329
Step 170, Loss 0.003310, PSNR 24.802067
Step 180, Loss 0.003309, PSNR 24.802778
Step 190, Loss 0.003309, PSNR 24.803457
Step 200, Loss 0.003308, PSNR 24.804113
Step 210, Loss 0.003308, PSNR 24.804741
Step 220, Loss 0.003307, PSNR 24.805346
Step 230, Loss 0.003307, PSNR 24.805927
Step 240, Loss 0.003306, PSNR 24.806482
Step 250, L

In [431]:
losses["siren_e"], psnrs["siren_e"], outputs["siren_e"], ress["siren_e"], _ = train(siren_e, total_steps, summary_steps, input_type="pe")

Step 0, Loss 0.012691, PSNR 18.965204
Step 10, Loss 0.004095, PSNR 23.877823
Step 20, Loss 0.003573, PSNR 24.470242
Step 30, Loss 0.003423, PSNR 24.655937
Step 40, Loss 0.003360, PSNR 24.736599
Step 50, Loss 0.003331, PSNR 24.774391
Step 60, Loss 0.003322, PSNR 24.786184
Step 70, Loss 0.003320, PSNR 24.789215
Step 80, Loss 0.003317, PSNR 24.793137
Step 90, Loss 0.003315, PSNR 24.795444
Step 100, Loss 0.003313, PSNR 24.797277
Step 110, Loss 0.003312, PSNR 24.798794
Step 120, Loss 0.003311, PSNR 24.800213
Step 130, Loss 0.003310, PSNR 24.801464
Step 140, Loss 0.003309, PSNR 24.802599
Step 150, Loss 0.003309, PSNR 24.803627
Step 160, Loss 0.003308, PSNR 24.804562
Step 170, Loss 0.003307, PSNR 24.805412
Step 180, Loss 0.003307, PSNR 24.806190
Step 190, Loss 0.003306, PSNR 24.806900
Step 200, Loss 0.003306, PSNR 24.807552
Step 210, Loss 0.003305, PSNR 24.808151
Step 220, Loss 0.003305, PSNR 24.808702
Step 230, Loss 0.003304, PSNR 24.809208
Step 240, Loss 0.003304, PSNR 24.809675
Step 250, L

In [432]:
losses["siren_pv"], psnrs["siren_pv"], outputs["siren_pv"], ress["siren_pv"], _ = train(siren_pv, total_steps, summary_steps, input_type="pv")

Step 0, Loss 0.018966, PSNR 17.220152
Step 10, Loss 0.003616, PSNR 24.418274
Step 20, Loss 0.002336, PSNR 26.315020
Step 30, Loss 0.001614, PSNR 27.920282
Step 40, Loss 0.001353, PSNR 28.686352
Step 50, Loss 0.001240, PSNR 29.064362
Step 60, Loss 0.001146, PSNR 29.408699
Step 70, Loss 0.001087, PSNR 29.638632
Step 80, Loss 0.001042, PSNR 29.822044
Step 90, Loss 0.001006, PSNR 29.972479
Step 100, Loss 0.000978, PSNR 30.098551
Step 110, Loss 0.000953, PSNR 30.207790
Step 120, Loss 0.000932, PSNR 30.304157
Step 130, Loss 0.000914, PSNR 30.390232
Step 140, Loss 0.000898, PSNR 30.467930
Step 150, Loss 0.000883, PSNR 30.538784
Step 160, Loss 0.000870, PSNR 30.603981
Step 170, Loss 0.000858, PSNR 30.664448
Step 180, Loss 0.000847, PSNR 30.720919
Step 190, Loss 0.000837, PSNR 30.773973
Step 200, Loss 0.000827, PSNR 30.824072
Step 210, Loss 0.000818, PSNR 30.871582
Step 220, Loss 0.000810, PSNR 30.916798
Step 230, Loss 0.000802, PSNR 30.959965
Step 240, Loss 0.000794, PSNR 31.001278
Step 250, L

In [ ]:
losses["siren_pd"], psnrs["siren_pd"], outputs["siren_pd"], ress["siren_pd"], _ = train(siren_pd, total_steps, summary_steps, input_type="pd")

Step 0, Loss 0.021368, PSNR 16.702330
Step 10, Loss 0.003722, PSNR 24.292135
Step 20, Loss 0.003399, PSNR 24.686518
Step 30, Loss 0.003140, PSNR 25.031012
Step 40, Loss 0.002918, PSNR 25.349466
Step 50, Loss 0.002816, PSNR 25.503670
Step 60, Loss 0.002790, PSNR 25.543192
Step 70, Loss 0.002762, PSNR 25.587135
Step 80, Loss 0.002748, PSNR 25.609369
Step 90, Loss 0.002736, PSNR 25.629410
Step 100, Loss 0.002726, PSNR 25.644960
Step 110, Loss 0.002717, PSNR 25.658859
Step 120, Loss 0.002709, PSNR 25.671156
Step 130, Loss 0.002703, PSNR 25.682339
Step 140, Loss 0.002696, PSNR 25.692657
Step 150, Loss 0.002690, PSNR 25.702259
Step 160, Loss 0.002685, PSNR 25.711269
Step 170, Loss 0.002679, PSNR 25.719780
Step 180, Loss 0.002674, PSNR 25.727873
Step 190, Loss 0.002670, PSNR 25.735607
Step 200, Loss 0.002665, PSNR 25.743034
Step 210, Loss 0.002661, PSNR 25.750195
Step 220, Loss 0.002656, PSNR 25.757126
Step 230, Loss 0.002652, PSNR 25.763863
Step 240, Loss 0.002648, PSNR 25.770426
Step 250, L

In [ ]:
losses["siren_pe"], psnrs["siren_pe"], outputs["siren_pe"], ress["siren_pe"], _ = train(siren_pv, total_steps, summary_steps, input_type="pe")

In [ ]:
losses["siren"], psnrs["siren"], outputs["siren"], ress["siren"], _ = train(siren, total_steps, summary_steps, input_type="pvde")

In [ ]:
losses["ff_siren_p"], psnrs["ff_siren_p"], outputs["ff_siren_p"], ress["ff_siren_p"], _ = train(ff_siren_p, total_steps, summary_steps, input_type="pv")

In [ ]:
losses["ff_siren_v"], psnrs["ff_siren_v"], outputs["ff_siren_v"], ress["ff_siren_v"], _ = train(ff_siren_v, total_steps, summary_steps, input_type="pv")

In [ ]:
losses["ff_siren_d"], psnrs["ff_siren_d"], outputs["ff_siren_d"], ress["ff_siren_d"], _ = train(ff_siren_d, total_steps, summary_steps, input_type="pd")

In [ ]:
losses["ff_siren_e"], psnrs["ff_siren_e"], outputs["ff_siren_e"], ress["ff_siren_e"], _ = train(ff_siren_e, total_steps, summary_steps, input_type="pe")

In [ ]:
losses["ff_siren_pv"], psnrs["ff_siren_pv"], outputs["ff_siren_pv"], ress["ff_siren_pv"], _ = train(ff_siren_pv, total_steps, summary_steps, input_type="pv")

In [ ]:
losses["ff_siren_pd"], psnrs["ff_siren_pd"], outputs["ff_siren_pd"], ress["ff_siren_pd"], _ = train(ff_siren_pd, total_steps, summary_steps, input_type="pd")

In [ ]:
losses["ff_siren_pe"], psnrs["ff_siren_pe"], outputs["ff_siren_pe"], ress["ff_siren_pe"], _ = train(ff_siren_pe, total_steps, summary_steps, input_type="pe")

In [ ]:
losses["ff_siren"], psnrs["ff_siren"], outputs["ff_siren"], ress["ff_siren"], _ = train(ff_siren, total_steps, summary_steps, input_type="pvde")

### FINER

In [ ]:
losses["finer_p"], psnrs["finer_p"], outputs["finer_p"], ress["finer_p"], _ = train(finer_p, total_steps, summary_steps, input_type="pv")

In [ ]:
losses["finer_v"], psnrs["finer_v"], outputs["finer_v"], ress["finer_v"], _ = train(finer_v, total_steps, summary_steps, input_type="pv")

In [ ]:
losses["finer_d"], psnrs["finer_d"], outputs["finer_d"], ress["finer_d"], _ = train(finer_d, total_steps, summary_steps, input_type="pd")

In [ ]:
losses["finer_e"], psnrs["finer_e"], outputs["finer_e"], ress["finer_e"], _ = train(finer_e, total_steps, summary_steps, input_type="pe")

In [ ]:
losses["finer_pv"], psnrs["finer_pv"], outputs["finer_pv"], ress["finer_pv"], _ = train(finer_pv, total_steps, summary_steps, input_type="pv")

In [ ]:
losses["finer_pd"], psnrs["finer_pd"], outputs["finer_pd"], ress["finer_pd"], _ = train(finer_pd, total_steps, summary_steps, input_type="pd")

In [ ]:
losses["finer_pe"], psnrs["finer_pe"], outputs["finer_pe"], ress["finer_pe"], _ = train(finer_pe, total_steps, summary_steps, input_type="pe")

In [ ]:
losses["finer"], psnrs["finer"], outputs["finer"], ress["finer"], _ = train(finer, total_steps, summary_steps, input_type="pvde")

In [ ]:
losses["ff_finer_p"], psnrs["ff_finer_p"], outputs["ff_finer_p"], ress["ff_finer_p"], _ = train(ff_finer_p, total_steps, summary_steps, input_type="pv")

In [ ]:
losses["ff_finer_v"], psnrs["ff_finer_v"], outputs["ff_finer_v"], ress["ff_finer_v"], _ = train(ff_finer_v, total_steps, summary_steps, input_type="pv")

In [ ]:
losses["ff_finer_d"], psnrs["ff_finer_d"], outputs["ff_finer_d"], ress["ff_finer_d"], _ = train(ff_finer_d, total_steps, summary_steps, input_type="pd")

In [ ]:
losses["ff_finer_e"], psnrs["ff_finer_e"], outputs["ff_finer_e"], ress["ff_finer_e"], _ = train(ff_finer_e, total_steps, summary_steps, input_type="pe")

In [ ]:
losses["ff_finer_pv"], psnrs["ff_finer_pv"], outputs["ff_finer_pv"], ress["ff_finer_pv"], _ = train(ff_finer_pv, total_steps, summary_steps, input_type="pv")

In [ ]:
losses["ff_finer_pd"], psnrs["ff_finer_pd"], outputs["ff_finer_pd"], ress["ff_finer_pd"], _ = train(ff_finer_pd, total_steps, summary_steps, input_type="pd")

In [ ]:
losses["ff_finer_pe"], psnrs["ff_finer_pe"], outputs["ff_finer_pe"], ress["ff_finer_pe"], _ = train(ff_finer_pe, total_steps, summary_steps, input_type="pe")

In [ ]:
losses["ff_finer"], psnrs["ff_finer"], outputs["ff_finer"], ress["ff_finer"], _ = train(ff_finer, total_steps, summary_steps, input_type="pvde")

### FINER20

In [ ]:
losses["finer20_p"], psnrs["finer20_p"], outputs["finer20_p"], ress["finer20_p"], _ = train(finer20_p, total_steps, summary_steps, input_type="pv")

In [ ]:
losses["finer20_v"], psnrs["finer20_v"], outputs["finer20_v"], ress["finer20_v"], _ = train(finer20_v, total_steps, summary_steps, input_type="pv")

In [ ]:
losses["finer20_d"], psnrs["finer20_d"], outputs["finer20_d"], ress["finer20_d"], _ = train(finer20_d, total_steps, summary_steps, input_type="pd")

In [ ]:
losses["finer20_e"], psnrs["finer20_e"], outputs["finer20_e"], ress["finer20_e"], _ = train(finer20_e, total_steps, summary_steps, input_type="pe")

In [ ]:
losses["finer20_pv"], psnrs["finer20_pv"], outputs["finer20_pv"], ress["finer20_pv"], _ = train(finer20_pv, total_steps, summary_steps, input_type="pv")

In [ ]:
losses["finer20_pd"], psnrs["finer20_pd"], outputs["finer20_pd"], ress["finer20_pd"], _ = train(finer20_pd, total_steps, summary_steps, input_type="pd")

In [ ]:
losses["finer20_pe"], psnrs["finer20_pe"], outputs["finer20_pe"], ress["finer20_pe"], _ = train(finer20_pe, total_steps, summary_steps, input_type="pe")

In [ ]:
losses["finer20"], psnrs["finer20"], outputs["finer20"], ress["finer20"], _ = train(finer20, total_steps, summary_steps, input_type="pvde")

In [ ]:
losses["ff_finer20_p"], psnrs["ff_finer20_p"], outputs["ff_finer20_p"], ress["ff_finer20_p"], _ = train(ff_finer20_p, total_steps, summary_steps, input_type="pv")

In [ ]:
losses["ff_finer20_v"], psnrs["ff_finer20_v"], outputs["ff_finer20_v"], ress["ff_finer20_v"], _ = train(ff_finer20_v, total_steps, summary_steps, input_type="pv")

In [ ]:
losses["ff_finer20_d"], psnrs["ff_finer20_d"], outputs["ff_finer20_d"], ress["ff_finer20_d"], _ = train(ff_finer20_d, total_steps, summary_steps, input_type="pd")

In [ ]:
losses["ff_finer20_e"], psnrs["ff_finer20_e"], outputs["ff_finer20_e"], ress["ff_finer20_e"], _ = train(ff_finer20_e, total_steps, summary_steps, input_type="pe")

In [ ]:
losses["ff_finer20_pv"], psnrs["ff_finer20_pv"], outputs["ff_finer20_pv"], ress["ff_finer20_pv"], _ = train(ff_finer20_pv, total_steps, summary_steps, input_type="pv")

In [ ]:
losses["ff_finer20_pd"], psnrs["ff_finer20_pd"], outputs["ff_finer20_pd"], ress["ff_finer20_pd"], _ = train(ff_finer20_pd, total_steps, summary_steps, input_type="pd")

In [ ]:
losses["ff_finer20_pe"], psnrs["ff_finer20_pe"], outputs["ff_finer20_pe"], ress["ff_finer20_pe"], _ = train(ff_finer20_pe, total_steps, summary_steps, input_type="pe")

In [ ]:
losses["ff_finer20"], psnrs["ff_finer20"], outputs["ff_finer20"], ress["ff_finer20"], _ = train(ff_finer20, total_steps, summary_steps, input_type="pvde")

### FIREN

In [ ]:
# losses["firen_p"], psnrs["firen_p"], outputs["firen_p"], ress["firen_p"], _ = train(firen_p, total_steps, summary_steps, input_type="pv")

In [ ]:
# losses["firen_v"], psnrs["firen_v"], outputs["firen_v"], ress["firen_v"], _ = train(firen_v, total_steps, summary_steps, input_type="pv")

In [ ]:
# losses["firen_d"], psnrs["firen_d"], outputs["firen_d"], ress["firen_d"], _ = train(firen_d, total_steps, summary_steps, input_type="pd")

In [ ]:
# losses["firen_e"], psnrs["firen_e"], outputs["firen_e"], ress["firen_e"], _ = train(firen_e, total_steps, summary_steps, input_type="pe")

In [ ]:
# losses["firen_pv"], psnrs["firen_pv"], outputs["firen_pv"], ress["firen_pv"], _ = train(firen_pv, total_steps, summary_steps, input_type="pv")

In [ ]:
# losses["firen_pd"], psnrs["firen_pd"], outputs["firen_pd"], ress["firen_pd"], _ = train(firen_pd, total_steps, summary_steps, input_type="pd")

In [ ]:
# losses["firen_pe"], psnrs["firen_pe"], outputs["firen_pe"], ress["firen_pe"], _ = train(firen_pe, total_steps, summary_steps, input_type="pe")

In [ ]:
# losses["firen"], psnrs["firen"], outputs["firen"], ress["firen"], _ = train(firen, total_steps, summary_steps, input_type="pvde")

In [ ]:
# losses["ff_firen_pv"], psnrs["ff_firen_pv"], outputs["ff_firen_pv"], ress["ff_firen_pv"], _ = train(ff_firen_pv, total_steps, summary_steps, input_type="pv")

In [ ]:
# losses["ff_firen_pd"], psnrs["ff_firen_pd"], outputs["ff_firen_pd"], ress["ff_firen_pd"], _ = train(ff_firen_pd, total_steps, summary_steps, input_type="pd")

In [ ]:
# losses["ff_firen_pe"], psnrs["ff_firen_pe"], outputs["ff_firen_pe"], ress["ff_firen_pe"], _ = train(ff_firen_pe, total_steps, summary_steps, input_type="pe")

In [ ]:
# losses["ff_firen"], psnrs["ff_firen"], outputs["ff_firen"], ress["ff_firen"], _ = train(ff_firen, total_steps, summary_steps, input_type="pvde")

## Visualize

In [ ]:
print("Final Loss:")

#plt.figure(figsize=fig_size)
plt.subplots(figsize=fig_size)
for n in losses:
	plt.plot(losses[n], label=n, linewidth=line_width)
	plt.legend(prop={"size": font_size})
	# print(f"{n}: {losses[n][-1]}")
plt.xlabel("Steps")
plt.ylabel("Loss")
# plt.ylim(0.0, 1.0)
plt.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
plt.legend(bbox_to_anchor=(1, 1), loc=1, borderaxespad=0)
plt.savefig(f"{output_dir}/loss_{total_steps}.jpg", dpi=100, bbox_inches="tight")

plt.clf()
#plt.figure(figsize=fig_size)
plt.subplots(figsize=fig_size)
for n in losses:
  plt.plot(losses[n], label=n, linewidth=line_width)
  plt.legend(prop={"size": font_size})
  # print(f"{n}: {losses[n][-1]}")
plt.xlabel("Steps")
plt.ylabel("Loss")
# plt.ylim(0.0, 0.013)
# plt.ylim(0.0, 0.5)
plt.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
plt.legend(bbox_to_anchor=(1, 1), loc=1, borderaxespad=0)
# plt.savefig(f"{output_dir}/loss_{total_steps}_small.jpg", dpi=100, bbox_inches="tight")

In [ ]:
print("Final PSNR:")

#plt.figure(figsize=fig_size)
plt.subplots(figsize=fig_size)
for n in psnrs:
  plt.plot(psnrs[n], label=n, linewidth=line_width)
  plt.legend(prop={"size": font_size})
  # Print best
  max_psnr  = max(psnrs[n])
  max_index = psnrs[n].index(max_psnr)
  print(f"{max_psnr}")
  # print(f"PSNR {n}: {max_psnr}")
  # print(f"PSNR {n}: {psnrs[n][-1]}")
plt.xlabel("Steps")
plt.ylabel("PSNR")
# plt.ylim(24, 28)
plt.ticklabel_format(axis="y", style="plain", scilimits=(0, 0))
# plt.legend(bbox_to_anchor=(1, 1), loc=1, borderaxespad=0)
plt.savefig(f"{output_dir}/psnr_{total_steps}.jpg", dpi=100, bbox_inches="tight")

## Save

In [ ]:
!pip install imageio-ffmpeg
import os, imageio

In [ ]:
# Save out video
if save_images:
    all_preds = np.concatenate([outputs[n] for n in outputs], axis=-1)
    data8     = (255 * (np.clip(all_preds, -1, 1) + 1) / 2).astype(np.uint8)
    f         = os.path.join(f"{output_dir}/training_convergence_{total_steps}.mp4")
    imageio.mimwrite(f, data8, fps=20)
    N = len(outputs)
    # Display video inline
    from IPython.display import HTML
    from base64 import b64encode

    mp4      = open(f, "rb").read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

In [ ]:
HTML(f'''
<video width=1000 controls autoplay loop>
      <source src="{data_url}" type="video/mp4">
</video>
<table width="1000" cellspacing="0" cellpadding="0">
  <tr>{''.join(N*[f'<td width="{1000//len(outputs)}"></td>'])}</tr>
  <tr>{''.join(N*['<td style="text-align:center">{}</td>'])}</tr>
</table>
'''.format(*list(outputs.keys())))

In [ ]:
plt.imshow(data8[-1], cmap="gray")

In [ ]:
image_v = image_v.squeeze(0).squeeze(0).cpu().detach().numpy()
ref_v   = ref_v.squeeze(0).squeeze(0).cpu().detach().numpy()
res_gt  = res_gt.squeeze(0).squeeze(0).cpu().detach().numpy()

image_v = np.around((image_v / 2 + 0.5) * 255).astype(np.uint8)
ref_v   = np.around((ref_v   / 2 + 0.5) * 255).astype(np.uint8)
res_gt  = np.around((res_gt  / 2 + 0.5) * 255).astype(np.uint8)
plt.imshow(ref_v, cmap="gray")
plt.title("groundtruth")

In [ ]:
def draw_figure(image, text, save_path, cmap="gray"):
	font = {
		# 'family': 'serif',
		"color" : "white",
		"weight": "bold",
		"size"  : 28,
	}
	fig, axs       = plt.subplots(1, 1)
	dpi            = 100
	left, width    = 0, 1
	bottom, height = 0, 1
	right          = left   + width
	center_x       = left   + width  / 2
	center_y       = bottom + height / 2
	top            = bottom + height
	p              = plt.Rectangle((left, bottom), width, height, linewidth=0, fill=False, facecolor="none", edgecolor=None)
	p.set_transform(axs.transAxes)
	p.set_clip_on(True)

	axs.add_patch(p)
	axs.imshow(image, cmap=cmap)
	axs.title.set_text("")
	axs.text(right - 0.01, bottom + 0.01, text,
			 horizontalalignment = "right",
			 verticalalignment   = "bottom",
			 transform           = axs.transAxes,
			 fontdict            = font)
	axs.set_xticks([])
	axs.set_yticks([])
	axs.set_axis_off()
	plt.show()
	fig.savefig(save_path, dpi=dpi, bbox_inches="tight", pad_inches=0)


if save_images:
	image_pil      = Image.fromarray(image_v)
	image_gradient = image_pil.filter(ImageFilter.FIND_EDGES)

	ref_pil        = Image.fromarray(ref_v)
	ref_gradient   = ref_pil.filter(ImageFilter.FIND_EDGES)

	res_pil        = Image.fromarray(res_gt)
	res_gradient   = res_pil.filter(ImageFilter.FIND_EDGES)

	# Debugging Purpose
	'''
	gray       = cv2.imread("data/51_ref_dav2_vitl_c.jpg", cv2.IMREAD_GRAYSCALE)
	ksize       = 3
	gX          = cv2.Sobel(gray, ddepth=cv2.CV_32F, dx=1, dy=0, ksize=ksize)
	gY          = cv2.Sobel(gray, ddepth=cv2.CV_32F, dx=0, dy=1, ksize=ksize)
	gX          = cv2.convertScaleAbs(gX)
	gY          = cv2.convertScaleAbs(gY)
	combined    = cv2.addWeighted(gX, 0.5, gY, 0.5, 0)
	gt_pil      = Image.fromarray(combined)
	gradient_gt = gt_pil.filter(ImageFilter.FIND_EDGES)
	'''

	# image_pil.save(f"{output_dir}/image_v.jpg")
	draw_figure(image_v, "", f"{output_dir}/image_v_color.jpg", cmap="viridis")
	# draw_figure(image_v, "", f"{output_dir}/image_v_gray.jpg")
	# image_gradient.save(f"{output_dir}/image_v_gradient.jpg")
	draw_figure(image_gradient, "", f"{output_dir}/image_v_gradient_color.jpg", cmap="viridis")
	# draw_figure(image_gradient, "", f"{output_dir}/image_v_gradient_gray.jpg")

	# ref_pil.save(f"{output_dir}/ref_v.jpg")
	draw_figure(ref_v, "", f"{output_dir}/ref_v_color.jpg", cmap="viridis")
	# draw_figure(ref_v, "", f"{output_dir}/ref_v_gray.jpg")
	# ref_gradient.save(f"{output_dir}/ref_v_gradient.jpg")
	draw_figure(ref_gradient, "", f"{output_dir}/ref_v_gradient_color.jpg", cmap="viridis")
	# draw_figure(ref_gradient, "", f"{output_dir}/ref_v_gradient_gray.jpg")

	# res_pil.save(f"{output_dir}/res.jpg")
	draw_figure(res_gt, "", f"{output_dir}/res_color.jpg", cmap="viridis")
	# draw_figure(res_gt, "", f"{output_dir}/res_gray.jpg")
	# res_gradient.save(f"{output_dir}/res_gradient.jpg")
	draw_figure(res_gradient, "", f"{output_dir}/res_gradient_color.jpg", cmap="viridis")
	# draw_figure(res_gradient, "", f"{output_dir}/res_gradient_gray.jpg")

	print("gradient mean squared error:")
	for i, n in enumerate(outputs):
		if n in ["ref"]:
			continue
		n_        = n.lower()
		max_psnr  = max(psnrs[n])
		max_index = psnrs[n].index(max_psnr)
		arr       = outputs[n][max_index]
		# arr       = outputs[n][-1]
		data8     = (255 * (np.clip(arr, -1, 1) + 1) / 2).astype(np.uint8)
		image     = Image.fromarray(data8)
		gradient  = image.filter(ImageFilter.FIND_EDGES)
		# print(f"{n}: {mse(gradient_gt, gradient)}")

		# image.save(f"{output_dir}/{n_}_v_{total_steps}.jpg")
		# draw_figure(image, f"PSNR: {psnrs[n][-1]:.2f}", f"{output_dir}/{n_}_v_gray_{total_steps}.jpg")
		draw_figure(image, f"PSNR: {psnrs[n][-1]:.2f}", f"{output_dir}/{n_}_v_color_{total_steps}.jpg", cmap="viridis")

		# gradient.save(f"{output_dir}/{n_}_v_gradient_{total_steps}.jpg")
		# draw_figure(gradient, f"PSNR: {psnrs[n][-1]:.2f}", f"{output_dir}/{n_}_v_gradient_gray_{total_steps}.jpg")
		draw_figure(gradient, f"PSNR: {psnrs[n][-1]:.2f}", f"{output_dir}/{n_}_v_gradient_color_{total_steps}.jpg", cmap="viridis")

	for i, n in enumerate(ress):
		if n in ["ref"]:
			continue
		n_        = n.lower()
		max_psnr  = max(psnrs[n])
		max_index = psnrs[n].index(max_psnr)
		arr       = ress[n][max_index]
		# arr       = ress[n][-1]
		data8     = (255 * (np.clip(arr, -1, 1) + 1) / 2).astype(np.uint8)
		image     = Image.fromarray(data8)

		# image.save(f"{output_dir}/{n_}_r_{total_steps}.jpg")
		# draw_figure(image, f"PSNR: {psnrs[n][-1]:.2f}", f"{output_dir}/{n_}_r_gray_{total_steps}.jpg")
		draw_figure(image, f"PSNR: {psnrs[n][-1]:.2f}", f"{output_dir}/{n_}_r_color_{total_steps}.jpg", cmap="viridis")